In [2]:
%pip install -q groq pydantic pymupdf python-docx rapidfuzz pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.1 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from groq import Groq
import os

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized successfully.")

Groq client initialized successfully.


In [4]:
from pydantic import BaseModel, Field
from typing import Literal

class RubricCriterion(BaseModel):
    name: str
    description: str
    weight: int = Field(ge=1, le=100)

class AssessmentQuestion(BaseModel):
    question: str
    topic: str
    question_type: Literal[
        "technical", "scenario", "problem_solving", "behavioral"
    ]
    difficulty: Literal["easy", "medium", "hard"]
    source_relevance: list[str]
    expected_answer_points: list[str]
    rubric: list[RubricCriterion]
    max_marks: int = Field(gt=0)

class QuestionSet(BaseModel):
    questions: list[AssessmentQuestion]

print("Question schema ready.")

Question schema ready.


In [140]:

import json
import re

MODEL_NAME = "openai/gpt-oss-20b"
print("Model:", MODEL_NAME)
print("Rubric required:", strict_schema["$defs"]["AssessmentQuestion"]["required"])
print(json.dumps(
    strict_schema["$defs"]["AssessmentQuestion"],
    indent=2
))

def redact_personal_details(text):
    """Remove common contact details before sending resume text to the LLM."""
    text = re.sub(
        r'\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b',
        '[EMAIL REDACTED]',
        text,
        flags=re.IGNORECASE
    )

    # Remove phone-like sequences, including Indian phone numbers.
    text = re.sub(
        r'(?<!\w)(?:\+?\d[\d\s().-]{8,}\d)(?!\w)',
        '[PHONE REDACTED]',
        text
    )

    # Remove common profile URLs
    text = re.sub(
        r'https?://(?:www\.)?(?:linkedin\.com|github\.com)/\S+',
        '[PROFILE URL REDACTED]',
        text,
        flags=re.IGNORECASE
    )

    return text


def generate_questions(
    resume_text="",
    job_description="",
    selected_topics=None,
    target_role="",
    experience_level="",
    total_questions=5,
    difficulty_distribution=None
):
    selected_topics = selected_topics or []

    difficulty_distribution = difficulty_distribution or {
        "easy": 2,
        "medium": 2,
        "hard": 1
    }

    if not (
        resume_text.strip()
        or job_description.strip()
        or selected_topics
    ):
        raise ValueError(
            "Provide a resume, job description, or selected topics."
        )

    if not 1 <= total_questions <= 15:
        raise ValueError("total_questions must be between 1 and 15.")

    if sum(difficulty_distribution.values()) != total_questions:
        raise ValueError(
            "Difficulty counts must sum to total_questions."
        )

    if any(
        level not in {"easy", "medium", "hard"} or count < 0
        for level, count in difficulty_distribution.items()
    ):
        raise ValueError("Invalid difficulty distribution.")

    safe_resume = redact_personal_details(resume_text)

    inputs = {
        "resume": safe_resume[:12000],
        "job_description": job_description[:12000],
        "selected_topics": selected_topics,
        "target_role": target_role,
        "experience_level": experience_level,
        "total_questions": total_questions,
        "difficulty_distribution": difficulty_distribution
    }

    system_prompt = """
You are a careful interview-question designer.

Generate original interview questions using the provided resume,
job description, selected topics, target role and experience level.

PERSONALIZATION:
- Use resume projects and skills only when they are present in the resume.
- Use the job description to identify role requirements.
- Cover selected topics as requested.
- Prefer questions relevant to both the role and candidate context.
- If the job description is only a role title, do not pretend it contains
  detailed requirements. Use the resume and selected topics as well.
- Do not ask for or repeat personal contact information.

QUALITY RULES:
- Never invent candidate skills, projects or experience.
- Do not treat a skill as proven merely because it appears in a job description.
- Follow the requested difficulty counts exactly.
- Include expected answer points for every question.
- Each rubric must contain meaningful criteria whose weights sum to 100.
- source_relevance must use only these labels:
  resume, job_description, selected_topics.
- A question must be answerable and appropriate for the target role.
- Avoid duplicate or near-duplicate questions within this batch.
- Return a JSON object with a top-level "questions" array.
- Each question must contain:
  question, topic, question_type, difficulty, source_relevance,
  expected_answer_points, rubric, max_marks.
- Each rubric criterion must contain name, description and weight.

STRICT OUTPUT RULES:
- question_type must be exactly one of:
  technical, scenario, problem_solving, behavioral.
- source_relevance must always be an array of strings.
- difficulty must be easy, medium or hard.
- Do not fabricate candidate facts.

Every question MUST include all these fields:
- question: string
- topic: string
- question_type: technical, scenario, problem_solving, or behavioral
- difficulty: easy, medium, or hard
- source_relevance: array of strings
- expected_answer_points: array of strings
- rubric: array of objects, each containing name, description, weight
- max_marks: positive number

Never omit rubric or max_marks.
Rubric weights must sum to 100.
Return valid JSON only.

STRICT OUTPUT REQUIREMENTS:
Every question object MUST include all required fields:
- question
- topic
- question_type
- difficulty
- source_relevance
- expected_answer_points
- rubric
- max_marks

The rubric must contain valid criteria with weights summing to 100.
max_marks must be a positive number.
Never omit rubric or max_marks.


MANDATORY OUTPUT REQUIREMENTS:
- Every question MUST include all required fields.
- Every question MUST include "rubric" as an array of rubric criterion objects.
- Every rubric criterion MUST include "name", "description", and "weight".
- Every question MUST include "max_marks" as a positive integer.
- Never omit "rubric" or "max_marks", even for a single question.
- Return only the JSON structure required by the supplied schema.
- Before returning, verify every question contains:
  question, topic, question_type, difficulty, source_relevance,
  expected_answer_points, rubric, max_marks.


"""
    print("Strict mode:", response_format if False else {
    "type": "json_schema",
    "json_schema": {
        "name": "question_set",
        "strict": True,
        "schema": strict_schema
    }
})

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": json.dumps(inputs, ensure_ascii=False)
                }
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "question_set",
                    "strict": True,
                    "schema": strict_schema
                }
            },
            temperature=0.2,
            max_completion_tokens=8000
        )
    except Exception as e:
        print("ERROR TYPE:", type(e).__name__)
        print("ERROR DETAILS:", str(e))
        raise

    print("RAW MODEL OUTPUT:", response.choices[0].message.content)

    raw_output = response.choices[0].message.content

    if not raw_output:
        raise ValueError("The model returned an empty response.")

    parsed = json.loads(raw_output)

    if not isinstance(parsed.get("questions"), list):
        raise ValueError("Model response must contain a questions array.")

    type_mapping = {
        "conceptual": "technical",
        "theory": "technical",
        "technical_theory": "technical",
        "situational": "scenario",
        "coding": "problem_solving",
        "hr": "behavioral"
    }

    for q in parsed["questions"]:
        source = q.get("source_relevance", [])

        if isinstance(source, str):
            source = [source]

        q["source_relevance"] = source

        qtype = q.get("question_type", "")
        if isinstance(qtype, str):
            qtype = qtype.strip().lower()
            q["question_type"] = type_mapping.get(qtype, qtype)




    # Normalize expected_answer_points
    for q in parsed["questions"]:
        points = q.get("expected_answer_points")

        if points is None:
            q["expected_answer_points"] = []

        elif isinstance(points, (int, float)):
            q["expected_answer_points"] = [str(points)]

        elif isinstance(points, str):
            q["expected_answer_points"] = [points.strip()] if points.strip() else []

        elif isinstance(points, list):
            normalized_points = []

            for point in points:
                if isinstance(point, str):
                    if point.strip():
                        normalized_points.append(point.strip())

                elif isinstance(point, dict):
                    point_text = point.get("point", "")
                    if point_text:
                        normalized_points.append(str(point_text).strip())

                elif isinstance(point, (int, float)):
                    normalized_points.append(str(point))

            q["expected_answer_points"] = normalized_points


    question_set = QuestionSet.model_validate(parsed)
    for q in question_set.questions:
        q.max_marks = 10

    if len(question_set.questions) != total_questions:
        raise ValueError(
            f"Expected {total_questions} questions, "
            f"received {len(question_set.questions)}. Please retry."
        )

    for q in question_set.questions:
        if sum(c.weight for c in q.rubric) != 100:
            raise ValueError(
                f"Rubric weights must total 100 for: {q.question}"
            )

        if q.max_marks <= 0 or not q.expected_answer_points:
            raise ValueError("Question failed quality validation.")

        if not q.topic.strip() or not q.question.strip():
            raise ValueError("Question or topic cannot be empty.")

    actual_counts = {
        level: sum(
            q.difficulty == level for q in question_set.questions
        )
        for level in ("easy", "medium", "hard")
    }

    expected_counts = {
        level: difficulty_distribution.get(level, 0)
        for level in ("easy", "medium", "hard")
    }

    if actual_counts != expected_counts:
        raise ValueError(
            f"Difficulty mismatch. Expected {expected_counts}, "
            f"received {actual_counts}. Please retry."
        )

    return question_set


Model: openai/gpt-oss-20b
Rubric required: ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks']
{
  "properties": {
    "question": {
      "title": "Question",
      "type": "string"
    },
    "topic": {
      "title": "Topic",
      "type": "string"
    },
    "question_type": {
      "enum": [
        "technical",
        "scenario",
        "problem_solving",
        "behavioral"
      ],
      "title": "Question Type",
      "type": "string"
    },
    "difficulty": {
      "enum": [
        "easy",
        "medium",
        "hard"
      ],
      "title": "Difficulty",
      "type": "string"
    },
    "source_relevance": {
      "items": {
        "type": "string"
      },
      "title": "Source Relevance",
      "type": "array"
    },
    "expected_answer_points": {
      "items": {
        "type": "string"
      },
      "title": "Expected Answer Points",
      "type": "array"
    },
    "rubric": {
      "it

In [141]:
import json

print("ROOT REQUIRED:")
print(strict_schema.get("required"))

print("\nROOT PROPERTIES:")
print(list(strict_schema.get("properties", {}).keys()))

question_schema = strict_schema["$defs"]["AssessmentQuestion"]

print("\nQUESTION REQUIRED:")
print(question_schema.get("required"))

print("\nQUESTION PROPERTIES:")
print(list(question_schema.get("properties", {}).keys()))

print("\nRUBRIC PROPERTY:")
print(json.dumps(
    question_schema.get("properties", {}).get("rubric"),
    indent=2
))

print("\nSTRICT SCHEMA VALIDATION:")
assert "rubric" in question_schema.get("required", [])
assert "max_marks" in question_schema.get("required", [])
assert "rubric" in question_schema.get("properties", {})
assert "max_marks" in question_schema.get("properties", {})
print("PASS: rubric and max_marks are required inside each question.")

ROOT REQUIRED:
['questions']

ROOT PROPERTIES:
['questions']

QUESTION REQUIRED:
['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks']

QUESTION PROPERTIES:
['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks']

RUBRIC PROPERTY:
{
  "items": {
    "$ref": "#/$defs/RubricCriterion"
  },
  "title": "Rubric",
  "type": "array"
}

STRICT SCHEMA VALIDATION:
PASS: rubric and max_marks are required inside each question.


In [142]:
def inspect_schema_objects(schema, path="root"):
    if not isinstance(schema, dict):
        return

    if schema.get("type") == "object":
        print(
            path,
            "| additionalProperties =",
            schema.get("additionalProperties", "MISSING")
        )

    for key, value in schema.items():
        if isinstance(value, dict):
            inspect_schema_objects(value, f"{path}.{key}")
        elif isinstance(value, list):
            for i, item in enumerate(value):
                inspect_schema_objects(item, f"{path}.{key}[{i}]")


inspect_schema_objects(strict_schema)

root | additionalProperties = False
root.$defs.AssessmentQuestion | additionalProperties = False
root.$defs.RubricCriterion | additionalProperties = False


In [91]:
result = generate_questions(
    selected_topics=["Java"],
    total_questions=1,
    difficulty_distribution={
        "easy": 1,
        "medium": 0,
        "hard": 0
    }
)

print(result)

Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [10]:
print(json.dumps(strict_schema, indent=2))

{
  "$defs": {
    "AssessmentQuestion": {
      "properties": {
        "question": {
          "title": "Question",
          "type": "string"
        },
        "topic": {
          "title": "Topic",
          "type": "string"
        },
        "question_type": {
          "enum": [
            "technical",
            "scenario",
            "problem_solving",
            "behavioral"
          ],
          "title": "Question Type",
          "type": "string"
        },
        "difficulty": {
          "enum": [
            "easy",
            "medium",
            "hard"
          ],
          "title": "Difficulty",
          "type": "string"
        },
        "source_relevance": {
          "items": {
            "type": "string"
          },
          "title": "Source Relevance",
          "type": "array"
        },
        "expected_answer_points": {
          "items": {
            "type": "string"
          },
          "title": "Expected Answer Points",
          "type": 

In [11]:
test_schema = {
    "type": "object",
    "properties": {
        "question": {"type": "string"},
        "rubric": {
            "type": "array",
            "items": {"type": "string"}
        },
        "max_marks": {"type": "integer"}
    },
    "required": ["question", "rubric", "max_marks"],
    "additionalProperties": False
}

test = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Create one Java interview question. Return the required fields."
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "schema_test",
            "strict": True,
            "schema": test_schema
        }
    },
    max_completion_tokens=500
)

print(test.choices[0].message.content)

{"question":"Explain the concept of polymorphism in Java and provide an example illustrating compile-time and runtime polymorphism.","rubric":["Definition of polymorphism","Distinction between compile-time and runtime polymorphism","Example code demonstrating both","Explanation of method overriding vs overloading","Clarity and completeness"],"max_marks":10}


In [14]:
test_response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Create one Java interview question about try-with-resources."
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "question_test",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                    "rubric": {"type": "array", "items": {"type": "string"}},
                    "max_marks": {"type": "integer"}
                },
                "required": ["question", "rubric", "max_marks"],
                "additionalProperties": False
            }
        }
    },
    max_completion_tokens=1000
)

print(test_response.choices[0].message.content)

{"question":"Explain how the try-with-resources statement works in Java and provide an example that reads a file using BufferedReader.","rubric":["Describe the syntax of try-with-resources.","Explain automatic resource management.","Show example with BufferedReader reading file line by line."],"max_marks":5}


In [16]:
test_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": """
            Generate exactly one Java interview question.
            Include every field required by the supplied schema.
            Use one rubric criterion with weight 100.
            """
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "question_set",
            "strict": True,
            "schema": strict_schema
        }
    },
    max_completion_tokens=2000
)

print(test_response.choices[0].message.content)

{"questions":[{"question":"Explain how Java's memory model ensures thread safety with synchronization and volatile variables.","topic":"Java Concurrency","question_type":"technical","difficulty":"medium","source_relevance":["Java Language Specification","Java Concurrency in Practice"],"expected_answer_points":["Java Memory Model defines visibility of writes across threads.","Synchronization ensures mutual exclusion and establishes happens-before relations.","Volatile variables guarantee visibility of changes across threads but do not provide mutual exclusion."],"rubric":[{"name":"Comprehensive Understanding","description":"Demonstrates comprehensive understanding of Java Memory Model and synchronization mechanisms.","weight":100}],"max_marks":10}]}


In [17]:
print(json.dumps(strict_schema, indent=2))

{
  "$defs": {
    "AssessmentQuestion": {
      "properties": {
        "question": {
          "title": "Question",
          "type": "string"
        },
        "topic": {
          "title": "Topic",
          "type": "string"
        },
        "question_type": {
          "enum": [
            "technical",
            "scenario",
            "problem_solving",
            "behavioral"
          ],
          "title": "Question Type",
          "type": "string"
        },
        "difficulty": {
          "enum": [
            "easy",
            "medium",
            "hard"
          ],
          "title": "Difficulty",
          "type": "string"
        },
        "source_relevance": {
          "items": {
            "type": "string"
          },
          "title": "Source Relevance",
          "type": "array"
        },
        "expected_answer_points": {
          "items": {
            "type": "string"
          },
          "title": "Expected Answer Points",
          "type": 

In [7]:

import json

schema = QuestionSet.model_json_schema()

def make_strict(schema):
    """Make object schemas compatible with strict structured output."""
    if isinstance(schema, dict):
        if schema.get("type") == "object":
            schema["additionalProperties"] = False
            schema["required"] = list(schema.get("properties", {}).keys())

        for value in schema.values():
            make_strict(value)

    elif isinstance(schema, list):
        for item in schema:
            make_strict(item)

    return schema

strict_schema = make_strict(schema)

print(json.dumps(strict_schema, indent=2))


{
  "$defs": {
    "AssessmentQuestion": {
      "properties": {
        "question": {
          "title": "Question",
          "type": "string"
        },
        "topic": {
          "title": "Topic",
          "type": "string"
        },
        "question_type": {
          "enum": [
            "technical",
            "scenario",
            "problem_solving",
            "behavioral"
          ],
          "title": "Question Type",
          "type": "string"
        },
        "difficulty": {
          "enum": [
            "easy",
            "medium",
            "hard"
          ],
          "title": "Difficulty",
          "type": "string"
        },
        "source_relevance": {
          "items": {
            "type": "string"
          },
          "title": "Source Relevance",
          "type": "array"
        },
        "expected_answer_points": {
          "items": {
            "type": "string"
          },
          "title": "Expected Answer Points",
          "type": 

In [18]:
from pydantic import BaseModel, Field
from typing import Literal

class RubricCriterion(BaseModel):
    name: str
    description: str
    weight: int = Field(ge=1, le=100)

class AssessmentQuestion(BaseModel):
    question: str
    topic: str
    question_type: Literal[
        "technical", "scenario", "problem_solving", "behavioral"
    ]
    difficulty: Literal["easy", "medium", "hard"]
    source_relevance: list[str]
    expected_answer_points: list[str]
    rubric: list[RubricCriterion]
    max_marks: int = Field(gt=0)

class QuestionSet(BaseModel):
    questions: list[AssessmentQuestion]

print("Question schemas defined successfully!")

Question schemas defined successfully!


In [92]:
topics = input(
    "Enter topics separated by commas: "
).split(",")

role = input("Target role: ").strip()

result = generate_questions(
    selected_topics=[t.strip() for t in topics if t.strip()],
    target_role=role,
    experience_level="Entry-level",
    total_questions=5,
    difficulty_distribution={
        "easy": 2,
        "medium": 2,
        "hard": 1
    }
)

for i, q in enumerate(result.questions, start=1):
    print(f"\nQuestion {i}: {q.question}")
    print("Topic:", q.topic)
    print("Difficulty:", q.difficulty)
    print("Expected points:", q.expected_answer_points)
    print("Rubric:", [c.model_dump() for c in q.rubric])

Enter topics separated by commas: Java,DBMS,OOPs,DSA
Target role: Software Developer
Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficu

In [95]:

question_set_result = generate_questions(
    resume_text=globals().get("resume_text", ""),
    job_description=globals().get("job_description", ""),
    selected_topics=globals().get(
        "selected_topics", ["Java", "OOPS"]
    ),
    target_role=globals().get(
        "target_role", "Software Developer"
    ),
    experience_level="Entry-level",
    total_questions=5,
    difficulty_distribution={
        "easy": 2,
        "medium": 2,
        "hard": 1
    }
)

for i, q in enumerate(question_set_result.questions, start=1):
    print(f"\nQuestion {i}: {q.question}")
    print("Topic:", q.topic)
    print("Difficulty:", q.difficulty)
    print("Sources:", q.source_relevance)
    print("Expected points:", q.expected_answer_points)


Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [108]:
candidate_answer = """
Each LinkedList element is stored inside a separate node object.
Each node stores the element reference and references to the previous
and next nodes. These references and the object header require extra
memory compared with an ArrayList, which stores element references
in a backing array. A typical node may use around 24 bytes of overhead,
depending on JVM configuration. For very large lists, this increases
heap usage and garbage collection work. Nodes may also be scattered
in memory, reducing cache locality and traversal performance.
"""

evaluation = evaluate_answer(
    next_question,
    candidate_answer
)

print("Evaluation:")
print(evaluation)

HARD QUESTION EVALUATION:
{'criteria_scores': [{'criterion': 'Memory Estimation Accuracy', 'marks_awarded': 2.5, 'max_marks': 4.0, 'feedback': 'The answer correctly estimates the per-node overhead for a LinkedList node (24 bytes) and calculates the total for 10 million nodes. However, it omits the memory used by the Integer objects themselves and the ArrayList’s array header and capacity overhead, which are essential for a complete comparison. The missing details reduce the accuracy of the estimation.'}, {'criterion': 'Performance Analysis Depth', 'marks_awarded': 1.5, 'max_marks': 3.0, 'feedback': 'The candidate discusses insertion and deletion complexities for both structures and notes cache locality for ArrayList. Nonetheless, the analysis lacks depth on traversal performance, the impact of node references on cache misses, and a thorough comparison of amortized costs. The discussion is therefore only partially complete.'}, {'criterion': 'Optimization Feasibility and Justification', 

In [106]:
decision = choose_next_action(
    next_question,
    evaluation,
    asked_topics=["Java"]
)

print("Adaptive decision:")
print(decision)

Adaptive decision:
{'action': 'INCREASE_DIFFICULTY', 'current_topic': 'Java', 'next_difficulty': 'hard', 'percentage': 100.0, 'strengths': ['Accurate description of node structure and overhead', 'Realistic overhead estimate', 'Clear discussion of memory, GC, and cache effects'], 'missing_concepts': [], 'asked_topics': ['Java'], 'reason': 'Strong answer; test deeper understanding.'}


In [107]:
next_hard_question = generate_next_with_history(
    decision=decision,
    question_history=question_history,
    resume_text=globals().get("resume_text", ""),
    job_description=globals().get("job_description", ""),
    target_role="Software Developer",
    experience_level="Beginner"
)

print("NEXT QUESTION:", next_hard_question.question)
print("TOPIC:", next_hard_question.topic)
print("DIFFICULTY:", next_hard_question.difficulty)
print("HISTORY COUNT:", len(question_history))

Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [109]:
candidate_answer = """
Assuming a typical 64-bit JVM with compressed references,
each LinkedList node may use approximately 24 bytes of node overhead.
For 10 million nodes, that is about 240 million bytes, or roughly
229 MiB, for nodes alone. This excludes the Integer objects and
the LinkedList object itself.

An ArrayList stores references in a backing array. With 4-byte
compressed references, 10 million references take about 40 MB
before array header and unused capacity. The Integer objects are
additional if they are not shared.

LinkedList can insert or delete in O(1) time once the node position
is already known, but finding a position takes O(n). ArrayList
supports O(1) indexed access and amortized O(1) append, while
inserting or deleting in the middle requires shifting elements.
ArrayList usually has better cache locality and lower memory overhead.

To reduce memory, I would first benchmark ArrayList with a suitable
initial capacity. If frequent operations are at both ends, I would
consider ArrayDeque. If arbitrary middle insertions and deletions
are essential, I would consider a specialized pooled-node structure,
but only if benchmarks justify its added complexity.
"""

hard_evaluation = evaluate_answer(
    next_hard_question,
    candidate_answer
)

print("HARD QUESTION EVALUATION:")
print(hard_evaluation)

HARD QUESTION EVALUATION:
{'criteria_scores': [{'criterion': 'Memory Estimation Accuracy', 'marks_awarded': 2.0, 'max_marks': 4.0, 'feedback': 'The answer gives a rough per-node size and a total for 10\u202fmillion nodes, but it omits the Integer objects themselves and the ArrayList’s array header and capacity overhead. The comparison is therefore incomplete, earning half the possible points.'}, {'criterion': 'Performance Analysis Depth', 'marks_awarded': 2.0, 'max_marks': 3.0, 'feedback': 'Insertion, deletion, and traversal complexities are mentioned, and cache locality is briefly noted. However, the discussion lacks depth on traversal performance, cache line effects, and the impact of node allocation on performance, so it receives two‑thirds of the points.'}, {'criterion': 'Optimization Feasibility and Justification', 'marks_awarded': 1.5, 'max_marks': 3.0, 'feedback': 'A strategy is proposed (ArrayDeque or pooled nodes) but the justification is vague and the concrete impact on memor

In [110]:
decision_hard = choose_next_action(
    next_hard_question,
    hard_evaluation,
    asked_topics=["Java"]
)

print("Adaptive decision after hard question:")
print(decision_hard)

Adaptive decision after hard question:
{'action': 'CLARIFY_CONCEPT', 'current_topic': 'Java', 'next_difficulty': 'hard', 'percentage': 55.0, 'strengths': ['Provides a rough memory estimate for LinkedList nodes', 'Discusses insertion, deletion, and traversal complexities for both structures', 'Proposes a strategy to reduce memory overhead'], 'missing_concepts': ['Integer object overhead', 'ArrayList array header and capacity overhead', 'LinkedList node reference overhead', 'Cache line effects on traversal', 'Memory fragmentation', 'Primitive array optimization'], 'asked_topics': ['Java'], 'reason': 'Partial understanding; validate missing concepts.'}


In [111]:
print("TOTAL QUESTIONS IN HISTORY:", len(question_history))
print("\nLAST 3 HISTORY ENTRIES:")

for item in question_history[-3:]:
    print(item)
    print("-" * 60)

TOTAL QUESTIONS IN HISTORY: 7

LAST 3 HISTORY ENTRIES:
{'question': 'Explain the difference between an ArrayList and a LinkedList in Java.', 'concept': 'ArrayList vs LinkedList', 'angle': 'Internal structure, time complexity, memory and use cases'}
------------------------------------------------------------
{'question': "Consider Java's LinkedList implementation. Explain why each node incurs additional memory overhead compared to an array element in an ArrayList, estimate the typical overhead per node, and discuss how this overhead can affect performance when working with very large lists.", 'concept': 'LinkedList node overhead', 'angle': 'Memory overhead per node vs array element'}
------------------------------------------------------------
{'question': "Consider Java's java.util.LinkedList implementation. Suppose you need to store 10 million elements of type Integer. Estimate the total memory consumption of the LinkedList, including node overhead, and compare it to an ArrayList sto

In [112]:
from collections import Counter

normalized_questions = [
    item["question"].strip().lower()
    for item in question_history
    if isinstance(item, dict) and "question" in item
]

counts = Counter(normalized_questions)
duplicates = [q for q, count in counts.items() if count > 1]

print("History entries:", len(question_history))
print("Unique questions:", len(set(normalized_questions)))
print("Exact duplicates:", duplicates)

History entries: 7
Unique questions: 7
Exact duplicates: []


In [113]:
test_pairs = [
    (
        "Explain the difference between an ArrayList and a LinkedList in Java.",
        "How does Java ArrayList differ from LinkedList in terms of storage and performance?"
    ),
    (
        "Explain the difference between an ArrayList and a LinkedList in Java.",
        "What is method overloading in Java?"
    )
]

print("SEMANTIC DUPLICATE TEST CASES")
print("-" * 60)

for i, (q1, q2) in enumerate(test_pairs, start=1):
    print(f"Test {i}")
    print("Question A:", q1)
    print("Question B:", q2)
    print("Expected:", "POTENTIAL DUPLICATE" if i == 1 else "NOT DUPLICATE")
    print()

SEMANTIC DUPLICATE TEST CASES
------------------------------------------------------------
Test 1
Question A: Explain the difference between an ArrayList and a LinkedList in Java.
Question B: How does Java ArrayList differ from LinkedList in terms of storage and performance?
Expected: POTENTIAL DUPLICATE

Test 2
Question A: Explain the difference between an ArrayList and a LinkedList in Java.
Question B: What is method overloading in Java?
Expected: NOT DUPLICATE



In [114]:
import importlib.util

for library in ["sentence_transformers", "sklearn"]:
    print(
        library,
        "INSTALLED" if importlib.util.find_spec(library) else "NOT INSTALLED"
    )

sentence_transformers INSTALLED
sklearn INSTALLED


In [115]:

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load a lightweight pretrained sentence-embedding model
similarity_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

test_pairs = [
    (
        "Explain the difference between an ArrayList and a LinkedList in Java.",
        "How does Java ArrayList differ from LinkedList in terms of storage and performance?",
        True
    ),
    (
        "Explain the difference between an ArrayList and a LinkedList in Java.",
        "What is method overloading in Java?",
        False
    )
]

for i, (q1, q2, expected_duplicate) in enumerate(test_pairs, start=1):
    embeddings = similarity_model.encode([q1, q2])
    score = cosine_similarity(
        [embeddings[0]], [embeddings[1]]
    )[0][0]

    print(f"\nTest {i}")
    print(f"Similarity score: {score:.4f}")
    print(f"Expected duplicate: {expected_duplicate}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Test 1
Similarity score: 0.8658
Expected duplicate: True

Test 2
Similarity score: 0.4061
Expected duplicate: False


In [116]:

SEMANTIC_DUPLICATE_THRESHOLD = 0.80

def find_semantic_duplicate(candidate_question, history, threshold=SEMANTIC_DUPLICATE_THRESHOLD):
    history_questions = [
        item["question"]
        for item in history
        if isinstance(item, dict) and item.get("question")
    ]

    if not history_questions:
        return {
            "is_duplicate": False,
            "similarity": None,
            "matched_question": None
        }

    candidate_embedding = similarity_model.encode(
        [candidate_question],
        normalize_embeddings=True
    )

    history_embeddings = similarity_model.encode(
        history_questions,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        candidate_embedding,
        history_embeddings
    )[0]

    best_index = int(similarities.argmax())
    best_score = float(similarities[best_index])

    return {
        "is_duplicate": best_score >= threshold,
        "similarity": round(best_score, 4),
        "matched_question": history_questions[best_index]
    }


# Test only; does not append to or modify question_history
for q in [
    "How does Java ArrayList differ from LinkedList in terms of storage and performance?",
    "What is method overloading in Java?"
]:
    result = find_semantic_duplicate(q, question_history)
    print("\nCandidate:", q)
    print("Duplicate detected:", result["is_duplicate"])
    print("Best similarity:", result["similarity"])
    print("Matched history question:", result["matched_question"])



Candidate: How does Java ArrayList differ from LinkedList in terms of storage and performance?
Duplicate detected: True
Best similarity: 0.8658
Matched history question: Explain the difference between an ArrayList and a LinkedList in Java.

Candidate: What is method overloading in Java?
Duplicate detected: False
Best similarity: 0.5109
Matched history question: What is a wrapper class in Java and provide an example of one?


In [98]:
current_question = question_set_result.questions[0]

print("QUESTION:", current_question.question)
print("MAX MARKS:", current_question.max_marks)


QUESTION: Explain the difference between an ArrayList and a LinkedList in Java.
MAX MARKS: 10


In [100]:

print("Current question:")
print(current_question.question)

print("\nEvaluation score:")
print(evaluation["score"], "/", evaluation["max_marks"])

print("\nQuestion history:")
print(question_history)


Current question:
Explain the difference between an ArrayList and a LinkedList in Java.

Evaluation score:
8.5 / 10.0

Question history:
[{'question': 'What is a wrapper class in Java and provide an example of one?', 'concept': 'Wrapper class', 'angle': 'Definition and example'}, {'question': 'Which wrapper class in Java is used to represent a boolean value, and what is its primary purpose?', 'concept': 'Boolean wrapper class', 'angle': 'Primary purpose of Boolean wrapper'}, {'question': 'What is the default value of a Boolean object in Java when it is declared but not initialized?', 'concept': 'Boolean default value', 'angle': 'Default initialization of Boolean objects'}, {'question': 'Is the Boolean wrapper class immutable in Java? Explain why or why not.', 'concept': 'Boolean wrapper class', 'angle': 'Immutability and design rationale'}]


In [101]:

current_text = current_question.question

already_exists = any(
    item.get("question", "").strip().lower()
    == current_text.strip().lower()
    for item in question_history
)

if not already_exists:
    question_history.append({
        "question": current_text,
        "concept": "ArrayList vs LinkedList",
        "angle": "Internal structure, time complexity, memory and use cases"
    })

print("History updated.")
print("Total history entries:", len(question_history))
print("Latest history entry:")
print(question_history[-1])


History updated.
Total history entries: 5
Latest history entry:
{'question': 'Explain the difference between an ArrayList and a LinkedList in Java.', 'concept': 'ArrayList vs LinkedList', 'angle': 'Internal structure, time complexity, memory and use cases'}


In [90]:
print("Type:", type(result))
print("Fields:", result.model_fields.keys())

Type: <class '__main__.QuestionSet'>
Fields: dict_keys(['questions'])


/tmp/ipykernel_779/4079699018.py:2: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print("Fields:", result.model_fields.keys())


In [93]:
# Normalize list fields if the model returns a string
for field in ["strengths", "missing_concepts"]:
    value = result.get(field)

    if isinstance(value, str):
        result[field] = [value] if value.strip() else []
    elif value is None:
        result[field] = []
    elif not isinstance(value, list):
        raise ValueError(f"{field} must be a list or string.")

# Normalize feedback to a string
feedback = result.get("improvement_feedback", "")
if isinstance(feedback, list):
    result["improvement_feedback"] = " ".join(feedback)
elif not isinstance(feedback, str):
    raise ValueError("improvement_feedback must be a string.")

AttributeError: 'QuestionSet' object has no attribute 'get'

In [37]:
candidate_answer_2 = """
A hash table stores key-value pairs using a hash function to find
an index. Collisions can be handled using chaining or open addressing.
A browser can use caching to store frequently accessed data.
"""

evaluation_2 = evaluate_answer(
    question=result.questions[1],
    candidate_answer=candidate_answer_2
)

print(evaluation_2)

{'criteria_scores': [{'criterion': 'Definition Accuracy', 'marks_awarded': 0.0, 'max_marks': 4.0, 'feedback': 'The answer does not contain any definition of polymorphism; it discusses hash tables and caching instead.'}, {'criterion': 'Example Relevance', 'marks_awarded': 0.0, 'max_marks': 3.0, 'feedback': 'No example related to polymorphism or Java is provided; the content is unrelated to the question.'}, {'criterion': 'Explanation Depth', 'marks_awarded': 0.0, 'max_marks': 3.0, 'feedback': 'The answer lacks any explanation of how polymorphism works; it is entirely off-topic.'}], 'strengths': [], 'missing_concepts': ['polymorphism'], 'improvement_feedback': 'To improve, provide a correct definition of polymorphism, give a simple Java example that demonstrates method overriding or interface implementation, and explain the underlying mechanism in sufficient depth.', 'score': 0.0, 'max_marks': 10.0}


In [24]:

import json
import math

def evaluate_answer(question, candidate_answer):
    if not isinstance(candidate_answer, str) or not candidate_answer.strip():
        raise ValueError("Candidate answer cannot be empty.")

    rubric = question.rubric
    if not rubric:
        raise ValueError("Question must contain at least one rubric criterion.")

    total_weight = sum(item.weight for item in rubric)
    if total_weight <= 0:
        raise ValueError("Rubric weights must add up to a positive value.")

    max_marks = float(question.max_marks)

    # Allocate question marks proportionally across rubric criteria.
    criterion_limits = [
        max_marks * item.weight / total_weight
        for item in rubric
    ]

    criterion_names = [item.name for item in rubric]

    schema = {
        "type": "object",
        "properties": {
            "criteria_scores": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "criterion": {"type": "string"},
                        "marks_awarded": {"type": "number"},
                        "max_marks": {"type": "number"},
                        "feedback": {"type": "string"}
                    },
                    "required": [
                        "criterion",
                        "marks_awarded",
                        "max_marks",
                        "feedback"
                    ],
                    "additionalProperties": False
                }
            },
            "strengths": {
                "type": "array",
                "items": {"type": "string"}
            },
            "missing_concepts": {
                "type": "array",
                "items": {"type": "string"}
            },
            "improvement_feedback": {"type": "string"}
        },
        "required": [
            "criteria_scores",
            "strengths",
            "missing_concepts",
            "improvement_feedback"
        ],
        "additionalProperties": False
    }

    rubric_text = "\n".join(
        f"- {item.name}: {item.description}\n"
        f"  Maximum marks: {criterion_limits[i]:.4f}"
        for i, item in enumerate(rubric)
    )

    prompt = f"""
Evaluate the candidate answer fairly using the question and rubric.

QUESTION:
{question.question}

TOPIC: {question.topic}
DIFFICULTY: {question.difficulty}
TOTAL MAXIMUM MARKS: {max_marks}

RUBRIC:
{rubric_text}

CANDIDATE ANSWER:
{candidate_answer}

STRICT EVALUATION RULES:
1. Evaluate only the candidate's written answer.
2. Award marks according to correctness, completeness and the rubric.
3. Use every rubric criterion exactly once, in the original order.
4. Copy each criterion name exactly as given.
5. Use the exact maximum marks specified for each criterion.
6. marks_awarded must be between 0 and that criterion's maximum.
7. Give specific, useful feedback for EACH criterion.
8. Feedback must explain why marks were awarded or lost.
   Never return empty, generic, or placeholder feedback.
9. Do not assume the candidate knows something they did not write.
10. Identify missing concepts accurately; do not invent omissions.
11. Return only the JSON object matching the required schema.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,

        messages=[
            {
                "role": "system",
                "content": (
                    "You are a careful technical interviewer. "
                    "Grade consistently, objectively and according to "
                    "the supplied rubric. Never invent candidate content."
                )
            },
            {"role": "user", "content": prompt}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "answer_evaluation",
                "strict": True,
                "schema": schema
            }
        },
        max_completion_tokens=4096,
        temperature=0
    )

    content = response.choices[0].message.content
    if not content:
        raise ValueError("The model returned an empty evaluation.")

    result = json.loads(content)
    criteria = result["criteria_scores"]

    if len(criteria) != len(rubric):
        raise ValueError(
            f"Expected {len(rubric)} criteria, got {len(criteria)}."
        )

    total_score = 0.0

    for i, item in enumerate(criteria):
        expected_name = criterion_names[i]
        expected_max = criterion_limits[i]

        if item["criterion"] != expected_name:
            raise ValueError(
                f"Criterion mismatch at position {i + 1}: "
                f"expected '{expected_name}', got '{item['criterion']}'."
            )

        awarded = item["marks_awarded"]
        returned_max = item["max_marks"]
        feedback = item["feedback"].strip()

        if not math.isfinite(awarded) or not math.isfinite(returned_max):
            raise ValueError("Marks must be finite numbers.")

        if not math.isclose(
            returned_max, expected_max, rel_tol=0, abs_tol=0.02
        ):
            raise ValueError(
                f"Incorrect maximum marks for '{expected_name}'."
            )

        if awarded < 0 or awarded > expected_max + 0.02:
            raise ValueError(
                f"Marks out of range for '{expected_name}'."
            )

        if not feedback:
            raise ValueError(
                f"Missing feedback for '{expected_name}'."
            )

        # Normalize tiny floating-point boundary differences.
        item["max_marks"] = round(expected_max, 2)
        item["marks_awarded"] = round(
            min(max(awarded, 0), expected_max), 2
        )
        total_score += item["marks_awarded"]

    result["score"] = round(total_score, 2)
    result["max_marks"] = round(max_marks, 2)

    return result


In [38]:

test_evaluation = evaluate_answer(
    question=result.questions[1],
    candidate_answer=(
        "A hash table stores key-value pairs using a hash function. "
        "Collisions can be handled by chaining or open addressing. "
        "It can be used for fast lookups and caching."
    )
)

print(json.dumps(test_evaluation, indent=2, ensure_ascii=False))


{
  "criteria_scores": [
    {
      "criterion": "Definition Accuracy",
      "marks_awarded": 0,
      "max_marks": 4.0,
      "feedback": "The answer does not address polymorphism; it discusses hash tables instead."
    },
    {
      "criterion": "Example Relevance",
      "marks_awarded": 0,
      "max_marks": 3.0,
      "feedback": "No example of polymorphism is provided; the content is unrelated."
    },
    {
      "criterion": "Explanation Depth",
      "marks_awarded": 0,
      "max_marks": 3.0,
      "feedback": "The answer lacks any explanation of polymorphism or its mechanisms."
    }
  ],
  "strengths": [],
  "missing_concepts": [
    "polymorphism",
    "object-oriented programming",
    "Java example"
  ],
  "improvement_feedback": "The candidate should read the question carefully and provide a definition of polymorphism, illustrate it with a Java example, and explain how it works.",
  "score": 0.0,
  "max_marks": 10.0
}


In [39]:

# Grading consistency test — hash table question
question = result.questions[1]

test_cases = [
    {
        "label": "Strong answer",
        "answer": (
            "A hash table stores key-value pairs using a hash function "
            "to map keys to indexes. Collisions occur when two keys map "
            "to the same index and can be handled using chaining or open "
            "addressing. Java HashMap is a practical example used for "
            "fast key-based lookups."
        )
    },
    {
        "label": "Partial answer",
        "answer": (
            "A hash table stores key-value pairs. Collisions can be "
            "handled using chaining."
        )
    },
    {
        "label": "Weak answer",
        "answer": (
            "A hash table is a table that stores data. It is useful."
        )
    }
]

consistency_results = []

for case in test_cases:
    try:
        evaluation = evaluate_answer(
            question=question,
            candidate_answer=case["answer"]
        )

        consistency_results.append({
            "answer_type": case["label"],
            "score": evaluation["score"],
            "max_marks": evaluation["max_marks"],
            "percentage": round(
                100 * evaluation["score"] / evaluation["max_marks"], 2
            ),
            "status": "SUCCESS"
        })

    except Exception as e:
        consistency_results.append({
            "answer_type": case["label"],
            "status": "ERROR",
            "error": str(e)
        })

import pandas as pd
display(pd.DataFrame(consistency_results))


,answer_type,score,max_marks,percentage,status
0,Strong answer,0.0,10.0,0.0,SUCCESS
1,Partial answer,0.0,10.0,0.0,SUCCESS
2,Weak answer,0.0,10.0,0.0,SUCCESS


In [40]:

# Repeatability test: same answer, three independent evaluations
same_answer = test_cases[0]["answer"]
repeat_scores = []

for run_number in range(1, 4):
    evaluation = evaluate_answer(
        question=question,
        candidate_answer=same_answer
    )
    repeat_scores.append({
        "run": run_number,
        "score": evaluation["score"],
        "max_marks": evaluation["max_marks"]
    })

repeat_df = pd.DataFrame(repeat_scores)
display(repeat_df)

print("Score range:", round(
    repeat_df["score"].max() - repeat_df["score"].min(), 2
))


,run,score,max_marks
0,1,0.0,10.0
1,2,0.0,10.0
2,3,0.0,10.0


Score range: 0.0


In [43]:

test_evaluation = evaluate_answer(
    question=result.questions[1],
    candidate_answer=candidate_answers[1]
)

print(json.dumps(test_evaluation, indent=2, ensure_ascii=False))


{
  "criteria_scores": [
    {
      "criterion": "Definition Accuracy",
      "marks_awarded": 0,
      "max_marks": 4.0,
      "feedback": "The answer does not contain any definition of polymorphism; it discusses hash tables instead. Therefore no marks are awarded for definition accuracy."
    },
    {
      "criterion": "Example Relevance",
      "marks_awarded": 0,
      "max_marks": 3.0,
      "feedback": "No example related to polymorphism or Java is provided. The example given is about hash tables, which is irrelevant to the question."
    },
    {
      "criterion": "Explanation Depth",
      "marks_awarded": 0,
      "max_marks": 3.0,
      "feedback": "The answer lacks any explanation of how polymorphism works, its types, or its benefits. Consequently, zero marks are awarded."
    }
  ],
  "strengths": [],
  "missing_concepts": [
    "Polymorphism definition",
    "Java example demonstrating polymorphism",
    "Explanation of how polymorphism works"
  ],
  "improvement_feedba

In [44]:

# Evaluate the first generated question
selected_question = result.questions[0]

print("Question:", selected_question.question)
candidate_answer = input("\nEnter candidate answer: ")

evaluation = evaluate_answer(
    selected_question,
    candidate_answer
)

print("\n--- EVALUATION REPORT ---")
print(json.dumps(evaluation, indent=2, ensure_ascii=False))


Question: What are the primitive data types in Java and how do they differ from their corresponding wrapper classes?

Enter candidate answer: Java has eight built-in Primitive Data Types that store raw values directly, and each has a corresponding Wrapper Class in the java.lang package that turns that value into an object.  The 8 Primitive Types and Wrapper Classes  • byte \(\rightarrow \) Byte (8-bit signed integer) • short \(\rightarrow \) Short (16-bit signed integer) • int \(\rightarrow \) Integer (32-bit signed integer) • long \(\rightarrow \) Long (64-bit signed integer) • float \(\rightarrow \) Float (32-bit floating-point) • double \(\rightarrow \) Double (64-bit floating-point) • boolean \(\rightarrow \) Boolean (true or false) • char \(\rightarrow \) Character (16-bit Unicode character)

--- EVALUATION REPORT ---
{
  "criteria_scores": [
    {
      "criterion": "Correctness",
      "marks_awarded": 5.0,
      "max_marks": 5.0,
      "feedback": "The answer correctly lists al

In [45]:
%pip install -q pymupdf

In [46]:
import pymupdf
print("PyMuPDF installed successfully!")

PyMuPDF installed successfully!


In [47]:

from google.colab import files
import pymupdf

def extract_resume_text():
    uploaded = files.upload()

    if not uploaded:
        raise ValueError("Please upload a resume PDF.")

    filename, content = next(iter(uploaded.items()))

    if not filename.lower().endswith(".pdf"):
        raise ValueError("Please upload a PDF resume.")

    with pymupdf.open(stream=content, filetype="pdf") as doc:
        text = "\n".join(page.get_text() for page in doc)

    if not text.strip():
        raise ValueError(
            "No readable text found. This may be a scanned PDF."
        )

    return text.strip()

resume_text = extract_resume_text()
print("Resume extracted successfully.")
print(resume_text[:1000])


Saving Dipesh_Kumar_Resume (2).pdf to Dipesh_Kumar_Resume (2).pdf
Resume extracted successfully.
DIPESH KUMAR 
B.Tech Computer Science Engineering Student | Aspiring Software Developer 
Bhabhata, West Champaran, Bihar  |  9801338643  |  dipeshkumarvu98@gmail.com 
LinkedIn   |   GitHub   |   LeetCode 
PROFESSIONAL SUMMARY 
B.Tech Computer Science Engineering student at VFSTR with a 9.07/10 CGPA and hands-on experience building team-based, web-
oriented projects. Strong foundation in Data Structures & Algorithms (100+ LeetCode problems solved) and Web Development, with 
a growing interest in Artificial Intelligence and Agentic AI, pursued through project-based learning and development. 
EDUCATION 
Vignan's Foundation for Science, Technology & Research (VFSTR), Guntur, Andhra Pradesh 
B.Tech — Computer Science & Engineering | 3rd Semester | CGPA: 9.07/10 
Class XII — Bihar Board (Science) 
Percentage: 87.8% 
TECHNICAL SKILLS 
Programming Languages: Java, JavaScript, Python, C 
Web Develop

In [48]:
job_description = input("Paste Job Description: ").strip()
topics_input = input("Enter topics separated by commas: ").strip()
target_role = input("Target role: ").strip()

selected_topics = [
    topic.strip()
    for topic in topics_input.split(",")
    if topic.strip()
]

if not job_description:
    raise ValueError("Job Description cannot be empty.")

if not selected_topics:
    raise ValueError("Enter at least one topic.")

if not target_role:
    raise ValueError("Target role cannot be empty.")

print("\nInputs ready!")
print("Resume characters:", len(resume_text))
print("JD characters:", len(job_description))
print("Topics:", selected_topics)

Paste Job Description: Full-Time Software Developer
Enter topics separated by commas: Java,OOPs,DBMS,DSA
Target role: Software Developer

Inputs ready!
Resume characters: 3049
JD characters: 28
Topics: ['Java', 'OOPs', 'DBMS', 'DSA']


In [49]:
import json

candidate_answers = [
    # Q1: Abstract class vs interface
    """An abstract class can contain abstract and concrete methods.
    An interface defines a contract. I would use an abstract Vehicle
    class for shared speed and behavior, and a Flyable interface for
    both birds and airplanes.""",

    # Q2: Hash table
    """A hash table stores key-value pairs using a hash function.
    Collisions can be handled by chaining or open addressing.
    It can be used for fast lookups and caching.""",

    # Q3: ACID
    """ACID means Atomicity, Consistency, Isolation and Durability.
    Atomicity means all operations in a transaction succeed or none
    do. Transactions and rollback help maintain this property.""",

    # Q4: Deadlock
    """Deadlock occurs when processes wait for resources held by
    each other. Its four conditions are mutual exclusion, hold and
    wait, no preemption and circular wait. Resource ordering can
    help prevent it.""",

    # Q5: Performance bottlenecks
    """I would profile the application, inspect slow queries using
    EXPLAIN, add appropriate indexes, use connection pooling and
    cache frequently requested data with Redis. I would monitor
    performance before scaling services horizontally."""
]

if len(candidate_answers) != len(result.questions):
    raise ValueError("Answer count must match generated question count.")

assessment_report = []
total_score = 0.0
total_max_marks = 0.0

for i, (question, answer) in enumerate(
    zip(result.questions, candidate_answers), start=1
):
    evaluation = evaluate_answer(
        question=question,
        candidate_answer=answer
    )

    # Ensure each evaluation is a dictionary
    if not isinstance(evaluation, dict):
        raise TypeError(
            f"Question {i}: evaluate_answer must return a dictionary."
        )

    score = float(evaluation["score"])
    max_marks = float(evaluation["max_marks"])

    total_score += score
    total_max_marks += max_marks

    assessment_report.append({
        "question_number": i,
        "question": question.question,
        "topic": question.topic,
        "difficulty": question.difficulty,
        "evaluation": evaluation
    })

percentage = (
    (total_score / total_max_marks) * 100
    if total_max_marks else 0
)

final_report = {
    "total_questions": len(assessment_report),
    "total_score": round(total_score, 2),
    "total_max_marks": round(total_max_marks, 2),
    "percentage": round(percentage, 2),
    "question_results": assessment_report
}

print(json.dumps(final_report, indent=2, ensure_ascii=False))

{
  "total_questions": 5,
  "total_score": 0.0,
  "total_max_marks": 50.0,
  "percentage": 0.0,
  "question_results": [
    {
      "question_number": 1,
      "question": "What are the primitive data types in Java and how do they differ from their corresponding wrapper classes?",
      "topic": "Java",
      "difficulty": "easy",
      "evaluation": {
        "criteria_scores": [
          {
            "criterion": "Correctness",
            "marks_awarded": 0.0,
            "max_marks": 5.0,
            "feedback": "The answer does not mention any Java primitive data types or wrapper classes, so it is entirely incorrect."
          },
          {
            "criterion": "Clarity",
            "marks_awarded": 0.0,
            "max_marks": 3.0,
            "feedback": "The explanation is about abstract classes and interfaces, not about primitives or wrappers, so it lacks clarity on the requested topic."
          },
          {
            "criterion": "Completeness",
            "m

In [50]:

# Adaptive Engine - Step 1: Inspect existing output

print("Available variables:")
print([
    name for name in globals()
    if name in [
        "result", "evaluation", "eval_result",
        "evaluation_result", "question_set"
    ]
])

# Inspect generated question structure
if "result" in globals() and hasattr(result, "questions"):
    print("\nQUESTION STRUCTURE:")
    print(result.questions[0].model_dump())

# Inspect any existing evaluation result
for name in [
    "evaluation", "eval_result",
    "evaluation_result"
]:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name.upper()} STRUCTURE:")
        if hasattr(obj, "model_dump"):
            print(obj.model_dump())
        elif isinstance(obj, dict):
            print(obj)
        else:
            print(type(obj), obj)


Available variables:
['result', 'evaluation']

QUESTION STRUCTURE:
{'question': 'What are the primitive data types in Java and how do they differ from their corresponding wrapper classes?', 'topic': 'Java', 'question_type': 'technical', 'difficulty': 'easy', 'source_relevance': ['selected_topics'], 'expected_answer_points': ['List of primitive types (byte, short, int, long, float, double, char, boolean).', 'Explanation that wrapper classes (Byte, Short, Integer, Long, Float, Double, Character, Boolean) are objects that provide methods and can be used in collections.', 'Mention differences such as nullability, default values, and ability to use generics.'], 'rubric': [{'name': 'Correctness', 'description': 'Accurate identification of all primitive types and wrapper classes.', 'weight': 50}, {'name': 'Clarity', 'description': 'Clear explanation of differences and use cases.', 'weight': 30}, {'name': 'Completeness', 'description': 'All required points addressed.', 'weight': 20}], 'max_mar

In [51]:

def choose_next_action(question, evaluation, asked_topics=None):
    asked_topics = asked_topics or []

    # Support both dictionaries and Pydantic objects
    if hasattr(evaluation, "model_dump"):
        evaluation = evaluation.model_dump()
    elif not isinstance(evaluation, dict):
        evaluation = vars(evaluation)

    if hasattr(question, "model_dump"):
        question = question.model_dump()
    elif not isinstance(question, dict):
        question = vars(question)

    score = float(evaluation.get("score", 0))
    max_marks = float(evaluation.get("max_marks", 0))
    percentage = score / max_marks * 100 if max_marks else 0

    difficulty = question.get("difficulty", "easy").lower()

    if percentage >= 80:
        next_difficulty = {
            "easy": "medium",
            "medium": "hard",
            "hard": "hard"
        }.get(difficulty, "medium")

        action = "INCREASE_DIFFICULTY"
        reason = "Strong answer; test deeper understanding."

    elif percentage >= 50:
        next_difficulty = difficulty
        action = "CLARIFY_CONCEPT"
        reason = "Partial understanding; validate missing concepts."

    else:
        next_difficulty = "easy"
        action = "REVISIT_FUNDAMENTALS"
        reason = "Low score; check foundational understanding."

    return {
        "action": action,
        "current_topic": question.get("topic", "Unknown"),
        "next_difficulty": next_difficulty,
        "percentage": round(percentage, 2),
        "strengths": evaluation.get("strengths", []),
        "missing_concepts": evaluation.get("missing_concepts", []),
        "asked_topics": asked_topics,
        "reason": reason
    }


decision = choose_next_action(
    result.questions[0],
    evaluation,
    asked_topics=["Java, OOPS"]
)

print(decision)


{'action': 'REVISIT_FUNDAMENTALS', 'current_topic': 'Java', 'next_difficulty': 'easy', 'percentage': 0.0, 'strengths': [], 'missing_concepts': ['Thread safety definition', 'Synchronized keyword usage', 'Pitfalls and mitigation'], 'asked_topics': ['Java, OOPS'], 'reason': 'Low score; check foundational understanding.'}


In [52]:

import inspect

print(inspect.signature(generate_questions))


(resume_text='', job_description='', selected_topics=None, target_role='', experience_level='', total_questions=5, difficulty_distribution=None)


In [53]:

# Display the exact question that will be answered
current_question = result.questions[0]

print("QUESTION:")
print(current_question.question)

print("\nTOPIC:", current_question.topic)
print("DIFFICULTY:", current_question.difficulty)
print("MAX MARKS:", current_question.max_marks)


QUESTION:
What are the primitive data types in Java and how do they differ from their corresponding wrapper classes?

TOPIC: Java
DIFFICULTY: easy
MAX MARKS: 10


In [54]:

candidate_answer = """
An abstract class defines what an object is, providing shared state
and behavior. An interface defines what an object can do by specifying
a contract or capability.
"""

# Evaluate this answer against its actual question
evaluation = evaluate_answer(
    current_question,
    candidate_answer
)

if hasattr(evaluation, "model_dump"):
    evaluation = evaluation.model_dump()

print("EVALUATION:")
print(evaluation)

# Choose the next action using the matching question and evaluation
decision = choose_next_action(
    current_question,
    evaluation,
    asked_topics=["Java, OOPS"]
)

print("\nADAPTIVE DECISION:")
print(decision)


EVALUATION:
{'criteria_scores': [{'criterion': 'Correctness', 'marks_awarded': 0, 'max_marks': 5.0, 'feedback': 'The answer does not mention any primitive data types or wrapper classes, so it is entirely incorrect.'}, {'criterion': 'Clarity', 'marks_awarded': 0, 'max_marks': 3.0, 'feedback': 'The explanation is unrelated to the question and does not provide any clear information about Java primitives or wrappers.'}, {'criterion': 'Completeness', 'marks_awarded': 0, 'max_marks': 2.0, 'feedback': 'All required points are missing; the answer fails to address any part of the question.'}], 'strengths': [], 'missing_concepts': ['primitive data types', 'wrapper classes', 'differences between primitives and wrappers'], 'improvement_feedback': "The response should explicitly list Java's eight primitive types (byte, short, int, long, float, double, char, boolean) and their corresponding wrapper classes (Byte, Short, Integer, Long, Float, Double, Character, Boolean), and explain key differences s

In [55]:

# Build context from the real evaluation and adaptive decision
adaptive_context = f"""
Previous question: {current_question.question}
Previous topic: {current_question.topic}
Previous score: {decision['percentage']}%
Missing concepts: {decision['missing_concepts']}

Generate a different follow-up question.
Focus on the missing concepts listed above.
Difficulty: {decision['next_difficulty']}
Avoid repeating the previous question.
Include a useful rubric and expected answer points.
"""

# Generate exactly one next question
adaptive_result = generate_questions(
    resume_text=globals().get("resume_text", ""),
    job_description=adaptive_context,
    selected_topics=["Java", "OOPS"],
    target_role="Software Developer",
    experience_level="Beginner",
    total_questions=1,
    difficulty_distribution={
        "easy": 1 if decision["next_difficulty"] == "easy" else 0,
        "medium": 1 if decision["next_difficulty"] == "medium" else 0,
        "hard": 1 if decision["next_difficulty"] == "hard" else 0
    }
)

next_question = adaptive_result.questions[0]

print("NEXT ADAPTIVE QUESTION:")
print(next_question.question)
print("\nTOPIC:", next_question.topic)
print("DIFFICULTY:", next_question.difficulty)
print("\nWHY THIS QUESTION:")
print(decision["reason"])
print("Concepts to validate:", decision["missing_concepts"])


Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [56]:

candidate_answer_2 = """
Abstract classes can have constructors, mutable instance fields,
abstract methods and concrete methods. Interfaces cannot have
constructors, and their fields are public static final constants.
Interfaces can have default and static methods in modern Java.
A BankAccount abstract class can share balance and deposit logic,
while an Exportable interface can be implemented by unrelated classes.
A class can extend one class but implement multiple interfaces.
"""

evaluation_2 = evaluate_answer(
    next_question,
    candidate_answer_2
)

if hasattr(evaluation_2, "model_dump"):
    evaluation_2 = evaluation_2.model_dump()

print("EVALUATION:")
print(evaluation_2)

decision_2 = choose_next_action(
    next_question,
    evaluation_2,
    asked_topics=["Java, OOPS"]
)

print("\nNEXT ADAPTIVE DECISION:")
print(decision_2)


EVALUATION:
{'criteria_scores': [{'criterion': 'Correctness', 'marks_awarded': 0, 'max_marks': 4.0, 'feedback': 'The answer does not address the question about wrapper classes versus primitives; it discusses abstract classes and interfaces instead.'}, {'criterion': 'Depth of Explanation', 'marks_awarded': 0, 'max_marks': 2.5, 'feedback': 'No explanation of nullability, generics, or autoboxing is provided.'}, {'criterion': 'Example Relevance', 'marks_awarded': 0, 'max_marks': 2.0, 'feedback': 'No concrete example involving a wrapper class is given.'}, {'criterion': 'Clarity and Structure', 'marks_awarded': 1.0, 'max_marks': 1.5, 'feedback': 'The answer is presented in a clear and organized manner, but it does not address the question.'}], 'improvement_feedback': 'The response should directly discuss wrapper classes, their benefits over primitives, and provide a concrete example where a wrapper is required.', 'missing_concepts': ['wrapper classes', 'nullability', 'generics', 'autoboxing'

In [57]:

import json

def extract_question_metadata(question, topic):
    prompt = f"""
Analyze this interview question.

Question: {question}
Broad topic: {topic}

Return JSON with exactly these keys:
- concept: the specific concept being tested (2-5 words)
- angle: the exact knowledge angle being tested (3-8 words)

Examples:
Question: What is a resource leak and why is it harmful?
Output: {{"concept": "Resource leaks", "angle": "Definition and consequences"}}

Question: In what order are try-with-resources closed?
Output: {{"concept": "Try-with-resources", "angle": "Reverse resource closing order"}}

Use the actual question. Do not invent unrelated concepts.
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )

    data = json.loads(response.choices[0].message.content)

    return {
        "concept": str(data["concept"]).strip(),
        "angle": str(data["angle"]).strip()
    }

In [58]:

test_metadata = extract_question_metadata(
    "What is a resource leak in Java, and what are the consequences of not closing resources?",
    "Java"
)

print(test_metadata)


{'concept': 'Resource leaks', 'angle': 'Definition and consequences'}


In [173]:

# Track questions actually generated in this session
if "question_history" not in globals():
    question_history = []

print("Questions tracked:", len(question_history))


def generate_next_with_history(
    decision,
    question_history,
    resume_text="",
    job_description="",
    target_role="Software Developer",
    experience_level="Beginner"
):
    topic = decision["current_topic"]
    difficulty = decision["next_difficulty"]


    history_lines = []

    for item in question_history:
        if isinstance(item, dict):
            history_lines.append(
                f"- Question: {item['question']}\n"
                f"  Concept: {item.get('concept', 'Not recorded')}\n"
                f"  Tested angle: {item.get('angle', 'Not recorded')}"
            )
        else:
            # Supports old history entries that are plain strings
            history_lines.append(f"- Question: {item}")

    history_text = "\n".join(history_lines)


    context = f"""
Adaptive interview.
Topic: {topic}
Target difficulty: {difficulty}
Previous score: {decision['percentage']}%
Missing concepts: {decision['missing_concepts']}

Previously asked questions:
{history_text}

Generate exactly one new question.

Difficulty rules:
- easy: basic definition or simple explanation only.
- medium: practical application, comparison, or short code example.
- hard: multi-step reasoning, complex implementation, or edge cases.

The actual question complexity MUST match the requested difficulty.
If difficulty is easy, do NOT ask for multiple resources, manual exception
handling, suppressed exceptions, or complex code design.
If difficulty is medium, use a practical but manageable scenario.
If difficulty is hard, test deeper reasoning and edge cases.

Do not repeat or paraphrase previous questions.
Focus on missing concepts where relevant.
Return a valid question with rubric and expected answer points.


TOPIC CONTINUITY RULES:
- Stay on the same specific concept as the previous question.
- Increase difficulty by testing deeper reasoning, edge cases, or implementation of that concept.
- Do not jump to an unrelated Java concept or design pattern.
- Change subtopic only after the current concept has been sufficiently assessed.
- The question's topic label alone is not enough; its actual content must be relevant.

STRICT CONCEPT CONTINUITY:
- Continue with the exact concept tested by the current question.
- Do not switch to an unrelated concept within the same broad topic.
- Select a new, untested angle of the current concept.
- Match the requested difficulty to the adaptive decision.
- Use question history to avoid repeating previous questions.
- Do not assume a fixed concept or example from an earlier session.


SEMANTIC DUPLICATE PREVENTION:
- Read the full question history before generating a question.
- Do not generate a question that tests the same knowledge
  in nearly the same way as an earlier question.
- Compare meaning, not just exact wording.
- Stay on the same concept, but test a different aspect
  that has not already been tested.
- If no meaningful new angle is available, generate a
  deeper follow-up rather than rephrasing an old question.

  TOPIC AND CONCEPT RULES:
- Preserve the exact current subtopic.
- Keep the top-level topic unchanged.
- Use question history to select a genuinely different,
  untested angle of the same subtopic.
- Do not switch to another concept just because it is related.

"""

    distribution = {
        "easy": int(difficulty == "easy"),
        "medium": int(difficulty == "medium"),
        "hard": int(difficulty == "hard")
    }



    # Generate and validate up to 3 questions
    max_attempts = 3

    for attempt in range(max_attempts):
        generated = generate_questions(
            resume_text=resume_text,
            job_description=(job_description + "\n" + context).strip(),
            selected_topics=[topic],
            target_role=target_role,
            experience_level=experience_level,
            total_questions=1,
            difficulty_distribution=distribution
        )

        q = generated.questions[0]

        # Preserve top-level topic label
        # Preserve the topic generated by the model.
        # Reject only if the topic label is missing.
        if not q.topic or not q.topic.strip():
            print(f"Attempt {attempt + 1}: missing topic label. Retrying...")
            continue

        # Reject exact duplicate questions
        old_questions = {
    (
        old.get("question", "")
        if isinstance(old, dict)
        else old
    ).strip().casefold()
    for old in question_history
}

        if q.question.strip().casefold() in old_questions:
            print(f"Attempt {attempt + 1}: duplicate question. Retrying...")
            continue

        # Semantic duplicate check (runs locally; no API call)
        semantic_result = find_semantic_duplicate(
            q.question,
            question_history,
            threshold=SEMANTIC_DUPLICATE_THRESHOLD
        )

        if semantic_result["is_duplicate"]:
            print(
                f"Attempt {attempt + 1}: semantic duplicate "
                f"(similarity={semantic_result['similarity']}, "
                f"matched={semantic_result['matched_question']!r}). "
                "Retrying..."
            )
            continue


        # Technical accuracy check
        try:
            result = validate_question_technical_accuracy(q)
        except Exception as exc:
            raise RuntimeError(
                "Technical validation failed; question was not accepted."
            ) from exc

        if result.get("is_accurate") is True and not result.get("issues"):
            metadata = extract_question_metadata(q.question, topic)

            question_history.append({
                "question": q.question,
                "concept": metadata["concept"],
                "angle": metadata["angle"]
            })
            print(f"Question passed validation on attempt {attempt + 1}.")
            return q

        print(
            f"Attempt {attempt + 1}: question rejected.",
            result.get("issues", [])
        )

    raise RuntimeError(
        "Could not generate a unique, technically accurate question "
        "after 3 attempts. Please retry."
    )



print("\nHistory tracking is ready.")


Questions tracked: 8

History tracking is ready.


In [176]:
adaptive_result = run_adaptive_step(
    current_question=current_question,
    candidate_answer=candidate_answer,
    question_history=question_history,
    asked_topics=[],
    resume_text=resume_text,
    job_description=job_description,
    target_role=target_role,
    experience_level=experience_level
)

print("Current topic:", current_question.topic)
print("Next question:", adaptive_result["next_question"].question)
print("Next topic:", adaptive_result["next_question"].topic)
print("Next difficulty:", adaptive_result["next_question"].difficulty)

Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [175]:
experience_level = "Beginner"

In [146]:
import copy

history_backup = copy.deepcopy(question_history)
history_before = len(question_history)

try:
    result = generate_next_with_history(
        decision=decision,
        question_history=question_history,
        resume_text="",
        job_description="",
        target_role="Software Developer",
        experience_level="Beginner"
    )

    assert result is not None
    assert hasattr(result, "question")
    assert result.question.strip()
    assert len(question_history) == history_before + 1
    assert question_history[-1]["question"] == result.question

    final_check = find_semantic_duplicate(
        result.question,
        history_backup,
        threshold=SEMANTIC_DUPLICATE_THRESHOLD
    )

    assert not final_check["is_duplicate"], (
        f"Duplicate accepted: {final_check}"
    )

    print("PASS: Question generated successfully.")
    print("PASS: Question is not a semantic duplicate.")
    print("PASS: Question added to history.")
    print("Generated question:", result.question)

finally:
    question_history[:] = history_backup
    print("Original history restored:", len(question_history))

Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [147]:
import copy

history_backup = copy.deepcopy(question_history)

try:
    test_question = (
        "Explain Java exception handling using try, catch, "
        "finally, throw, and throws with a suitable example."
    )

    result = find_semantic_duplicate(
        test_question,
        question_history,
        threshold=SEMANTIC_DUPLICATE_THRESHOLD
    )

    print("Similarity:", result.get("similarity"))
    print("Duplicate:", result["is_duplicate"])

    assert result["is_duplicate"] is False, (
        "FAIL: An unrelated question was incorrectly flagged as duplicate."
    )

    print("PASS: Unrelated question is allowed.")
    print("PASS: Existing history was not modified.")

finally:
    question_history[:] = history_backup

Similarity: 0.4168
Duplicate: False
PASS: Unrelated question is allowed.
PASS: Existing history was not modified.


In [148]:
import inspect

functions_to_check = [
    "run_adaptive_step",
    "evaluate_answer",
    "choose_next_action",
    "generate_next_with_history",
    "generate_final_report",
    "build_final_report",
    "create_final_report",
]

for name in functions_to_check:
    fn = globals().get(name)
    if callable(fn):
        print(f"\n{name}{inspect.signature(fn)}")
    else:
        print(f"\n{name}: not defined")

print("\nRequired runtime variables:")
for name in [
    "question_history", "decision", "current_question",
    "candidate_answer", "asked_topics"
]:
    print(f"{name}: {'available' if name in globals() else 'missing'}")


run_adaptive_step(current_question, candidate_answer, question_history, asked_topics=None, resume_text='', job_description='', target_role='Software Developer', experience_level='Beginner')

evaluate_answer(question, candidate_answer)

choose_next_action(question, evaluation, asked_topics=None)

generate_next_with_history(decision, question_history, resume_text='', job_description='', target_role='Software Developer', experience_level='Beginner')

generate_final_report: not defined

build_final_report: not defined

create_final_report: not defined

Required runtime variables:
question_history: available
decision: available
current_question: available
candidate_answer: available
asked_topics: missing


In [149]:

import copy

history_backup = copy.deepcopy(question_history)

try:
    print("Current question:", current_question.question)
    print("Candidate answer:", candidate_answer)

    evaluation = evaluate_answer(
        current_question,
        candidate_answer
    )

    print("\nEvaluation result:")
    print(evaluation)

    decision_result = choose_next_action(
        current_question,
        evaluation,
        asked_topics=[]
    )

    print("\nAdaptive decision:")
    print(decision_result)

    assert evaluation is not None
    assert decision_result is not None

    print("\nPASS: Answer evaluation and adaptive decision completed.")

finally:
    question_history[:] = history_backup
    print("History restored:", len(question_history))



Current question: Explain the difference between an ArrayList and a LinkedList in Java.
Candidate answer: 
Assuming a typical 64-bit JVM with compressed references,
each LinkedList node may use approximately 24 bytes of node overhead.
For 10 million nodes, that is about 240 million bytes, or roughly
229 MiB, for nodes alone. This excludes the Integer objects and
the LinkedList object itself.

An ArrayList stores references in a backing array. With 4-byte
compressed references, 10 million references take about 40 MB
before array header and unused capacity. The Integer objects are
additional if they are not shared.

LinkedList can insert or delete in O(1) time once the node position
is already known, but finding a position takes O(n). ArrayList
supports O(1) indexed access and amortized O(1) append, while
inserting or deleting in the middle requires shifting elements.
ArrayList usually has better cache locality and lower memory overhead.

To reduce memory, I would first benchmark ArrayLi

In [150]:

import copy

history_backup = copy.deepcopy(question_history)

try:
    adaptive_result = run_adaptive_step(
        current_question=current_question,
        candidate_answer=candidate_answer,
        question_history=question_history,
        asked_topics=[],
        resume_text="",
        job_description="",
        target_role="Software Developer",
        experience_level="Beginner"
    )

    print("Adaptive step result:")
    print(adaptive_result)

    assert adaptive_result is not None
    print("\nPASS: Full adaptive step completed.")

finally:
    question_history[:] = history_backup
    print("History restored:", len(question_history))


Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [152]:
import inspect

for name in ["assessment_report", "final_report"]:
    obj = globals()[name]
    print(f"\n--- {name} ---")
    print("Type:", type(obj).__name__)

    if callable(obj):
        print("Signature:", inspect.signature(obj))
        try:
            print(inspect.getsource(obj)[:5000])
        except (OSError, TypeError) as e:
            print("Source unavailable:", e)
    else:
        print("Current value:", repr(obj)[:3000])


--- assessment_report ---
Type: list
Current value: [{'question_number': 1, 'question': 'What are the primitive data types in Java and how do they differ from their corresponding wrapper classes?', 'topic': 'Java', 'difficulty': 'easy', 'evaluation': {'criteria_scores': [{'criterion': 'Correctness', 'marks_awarded': 0.0, 'max_marks': 5.0, 'feedback': 'The answer does not mention any Java primitive data types or wrapper classes, so it is entirely incorrect.'}, {'criterion': 'Clarity', 'marks_awarded': 0.0, 'max_marks': 3.0, 'feedback': 'The explanation is about abstract classes and interfaces, not about primitives or wrappers, so it lacks clarity on the requested topic.'}, {'criterion': 'Completeness', 'marks_awarded': 0.0, 'max_marks': 2.0, 'feedback': 'None of the required points (primitive types, wrapper classes, differences) are addressed.'}], 'improvement_feedback': 'Provide a concise list of Java primitive types, their corresponding wrapper classes, and explain key differences su

In [153]:

from pprint import pprint

print("Assessment questions:", len(assessment_report))
print("Final report question count:", final_report.get("total_questions"))
print("Final report score:", final_report.get("total_score"))
print("Final report max marks:", final_report.get("total_max_marks"))

# Recalculate totals from the assessment records
earned = sum(
    float(item.get("evaluation", {}).get("score", 0))
    for item in assessment_report
)
maximum = sum(
    float(item.get("evaluation", {}).get("max_marks", 0))
    for item in assessment_report
)

print("\nRecalculated score:", earned)
print("Recalculated max marks:", maximum)
print("Recalculated percentage:", round(earned / maximum * 100, 2) if maximum else 0)

print("\nAssessment question numbers:")
pprint([item.get("question_number") for item in assessment_report])


Assessment questions: 5
Final report question count: 5
Final report score: 0.0
Final report max marks: 50.0

Recalculated score: 0.0
Recalculated max marks: 50.0
Recalculated percentage: 0.0

Assessment question numbers:
[1, 2, 3, 4, 5]


In [154]:
for item in assessment_report:
    evaluation = item.get("evaluation", {})
    print(
        f"Q{item.get('question_number')} | "
        f"Topic: {item.get('topic')} | "
        f"Score: {evaluation.get('score')}/"
        f"{evaluation.get('max_marks')}"
    )

Q1 | Topic: Java | Score: 0.0/10.0
Q2 | Topic: OOPs | Score: 0.0/10.0
Q3 | Topic: DBMS | Score: 0.0/10.0
Q4 | Topic: DSA | Score: 0.0/10.0
Q5 | Topic: Java | Score: 0.0/10.0


In [155]:

for item in assessment_report:
    ev = item.get("evaluation", {})
    print(f"\nQ{item.get('question_number')}: {item.get('question')}")
    print("Score:", ev.get("score"), "/", ev.get("max_marks"))
    print("Missing concepts:", ev.get("missing_concepts", []))
    print("Feedback:", ev.get("improvement_feedback", "N/A"))



Q1: What are the primitive data types in Java and how do they differ from their corresponding wrapper classes?
Score: 0.0 / 10.0
Missing concepts: ['primitive data types in Java', 'wrapper classes', 'differences between primitives and wrappers']
Feedback: Provide a concise list of Java primitive types, their corresponding wrapper classes, and explain key differences such as mutability, nullability, and use in collections.

Q2: Explain polymorphism in object-oriented programming and provide a simple example in Java.
Score: 0.0 / 10.0
Missing concepts: ['Polymorphism definition', 'Java example demonstrating polymorphism', 'Explanation of polymorphism mechanisms']
Feedback: To improve, the candidate should provide a clear definition of polymorphism, include a Java code example that demonstrates method overriding or interface implementation, and explain how polymorphism enables dynamic method dispatch.

Q3: Describe the ACID properties of a database transaction and explain why each proper

In [156]:
print("Latest adaptive evaluation:")
print(adaptive_result["evaluation"]["score"], "/",
      adaptive_result["evaluation"]["max_marks"])

print("\nReport records:", len(assessment_report))
print("Final report records:", final_report.get("total_questions"))


Latest adaptive evaluation:
10.0 / 10.0

Report records: 5
Final report records: 5


In [157]:

ip = get_ipython()

for keyword in ["assessment_report", "final_report", "question_results"]:
    print(f"\n--- Cells containing {keyword} ---")
    matches = [
        (i, cell)
        for i, cell in enumerate(ip.history_manager.input_hist_raw)
        if keyword in cell and (
            "=" in cell or "append" in cell or "update" in cell
        )
    ]
    for i, cell in matches[-5:]:
        print(f"\nCell {i}:\n{cell[:1800]}")



--- Cells containing assessment_report ---

Cell 152:
import inspect

for name in ["assessment_report", "final_report"]:
    obj = globals()[name]
    print(f"\n--- {name} ---")
    print("Type:", type(obj).__name__)

    if callable(obj):
        print("Signature:", inspect.signature(obj))
        try:
            print(inspect.getsource(obj)[:5000])
        except (OSError, TypeError) as e:
            print("Source unavailable:", e)
    else:
        print("Current value:", repr(obj)[:3000])

Cell 153:

from pprint import pprint

print("Assessment questions:", len(assessment_report))
print("Final report question count:", final_report.get("total_questions"))
print("Final report score:", final_report.get("total_score"))
print("Final report max marks:", final_report.get("total_max_marks"))

# Recalculate totals from the assessment records
earned = sum(
    float(item.get("evaluation", {}).get("score", 0))
    for item in assessment_report
)
maximum = sum(
    float(item.get("evaluatio

In [158]:
ip = get_ipython()
cell49 = ip.history_manager.input_hist_raw[49]
print(cell49[1500:5000])

   zip(result.questions, candidate_answers), start=1
):
    evaluation = evaluate_answer(
        question=question,
        candidate_answer=answer
    )

    # Ensure each evaluation is a dictionary
    if not isinstance(evaluation, dict):
        raise TypeError(
            f"Question {i}: evaluate_answer must return a dictionary."
        )

    score = float(evaluation["score"])
    max_marks = float(evaluation["max_marks"])

    total_score += score
    total_max_marks += max_marks

    assessment_report.append({
        "question_number": i,
        "question": question.question,
        "topic": question.topic,
        "difficulty": question.difficulty,
        "evaluation": evaluation
    })

percentage = (
    (total_score / total_max_marks) * 100
    if total_max_marks else 0
)

final_report = {
    "total_questions": len(assessment_report),
    "total_score": round(total_score, 2),
    "total_max_marks": round(total_max_marks, 2),
    "percentage": round(percentage, 2),
  

In [159]:

def append_adaptive_result_to_report(
    assessment_report,
    current_question,
    candidate_answer,
    evaluation
):
    """Append one evaluated adaptive question without overwriting prior records."""

    if not isinstance(evaluation, dict):
        raise TypeError("evaluation must be a dictionary")

    # Prevent accidentally appending the same question twice
    question_text = current_question.question.strip()
    if any(
        item.get("question", "").strip() == question_text
        for item in assessment_report
    ):
        raise ValueError("This question is already present in the report.")

    record = {
        "question_number": len(assessment_report) + 1,
        "question": question_text,
        "topic": current_question.topic,
        "difficulty": current_question.difficulty,
        "candidate_answer": candidate_answer,
        "evaluation": evaluation
    }

    assessment_report.append(record)
    return record


print("Helper defined. Existing report unchanged:", len(assessment_report))


Helper defined. Existing report unchanged: 5


In [160]:

# Add the latest adaptive result to the existing report
new_record = append_adaptive_result_to_report(
    assessment_report=assessment_report,
    current_question=current_question,
    candidate_answer=candidate_answer,
    evaluation=adaptive_result["evaluation"]
)

# Recalculate final report from all assessment records
total_score = sum(
    float(item["evaluation"]["score"])
    for item in assessment_report
)
total_max_marks = sum(
    float(item["evaluation"]["max_marks"])
    for item in assessment_report
)

final_report = {
    "total_questions": len(assessment_report),
    "total_score": round(total_score, 2),
    "total_max_marks": round(total_max_marks, 2),
    "percentage": round(
        total_score / total_max_marks * 100, 2
    ) if total_max_marks else 0,
    "question_results": assessment_report
}

print("Added question:", new_record["question"])
print("Question number:", new_record["question_number"])
print("Updated report:", final_report["total_questions"], "questions")
print("Score:", final_report["total_score"], "/", final_report["total_max_marks"])
print("Percentage:", final_report["percentage"], "%")


Added question: Explain the difference between an ArrayList and a LinkedList in Java.
Question number: 6
Updated report: 6 questions
Score: 10.0 / 60.0
Percentage: 16.67 %


In [161]:

latest = final_report["question_results"][-1]

print("Question count:", final_report["total_questions"])
print("Latest question:", latest["question"])
print("Candidate answer saved:", bool(latest.get("candidate_answer")))
print("Latest score:", latest["evaluation"]["score"])
print("Latest max marks:", latest["evaluation"]["max_marks"])

assert final_report["total_questions"] == 6
assert latest["evaluation"]["score"] == 10.0
assert latest.get("candidate_answer"), "Candidate answer missing from report"

print("\nPASS: Adaptive evaluation is included in the final report.")


Question count: 6
Latest question: Explain the difference between an ArrayList and a LinkedList in Java.
Candidate answer saved: True
Latest score: 10.0
Latest max marks: 10.0

PASS: Adaptive evaluation is included in the final report.


In [162]:

import json
from collections import defaultdict
from pathlib import Path

# Recalculate totals from the current assessment records
records = final_report["question_results"]

total_score = sum(float(r["evaluation"].get("score", 0)) for r in records)
total_max = sum(float(r["evaluation"].get("max_marks", 0)) for r in records)

topic_stats = defaultdict(lambda: {
    "questions": 0,
    "score": 0.0,
    "max_marks": 0.0,
    "strengths": [],
    "missing_concepts": [],
    "improvement_suggestions": []
})

all_strengths = []
all_missing = []
all_suggestions = []

for record in records:
    evaluation = record.get("evaluation", {})
    topic = record.get("topic") or "Unspecified"
    stats = topic_stats[topic]

    stats["questions"] += 1
    stats["score"] += float(evaluation.get("score", 0))
    stats["max_marks"] += float(evaluation.get("max_marks", 0))

    strengths = evaluation.get("strengths", [])
    missing = evaluation.get("missing_concepts", [])
    suggestion = evaluation.get("improvement_feedback", "")

    stats["strengths"].extend(strengths)
    stats["missing_concepts"].extend(missing)

    if suggestion:
        stats["improvement_suggestions"].append(suggestion)

    all_strengths.extend(strengths)
    all_missing.extend(missing)

    if suggestion:
        all_suggestions.append({
            "question_number": record.get("question_number"),
            "topic": topic,
            "suggestion": suggestion
        })

def unique(items):
    result = []
    seen = set()
    for item in items:
        key = str(item).strip().casefold()
        if key and key not in seen:
            seen.add(key)
            result.append(item)
    return result

topic_summary = {}

for topic, stats in topic_stats.items():
    topic_summary[topic] = {
        "questions": stats["questions"],
        "score": round(stats["score"], 2),
        "max_marks": round(stats["max_marks"], 2),
        "percentage": round(
            stats["score"] / stats["max_marks"] * 100, 2
        ) if stats["max_marks"] else 0,
        "strengths": unique(stats["strengths"]),
        "missing_concepts": unique(stats["missing_concepts"]),
        "improvement_suggestions": unique(stats["improvement_suggestions"])
    }

# Enrich final report while preserving question-level records
final_report.update({
    "total_questions": len(records),
    "total_score": round(total_score, 2),
    "total_max_marks": round(total_max, 2),
    "percentage": round(total_score / total_max * 100, 2) if total_max else 0,
    "topic_summary": topic_summary,
    "overall_strengths": unique(all_strengths),
    "overall_missing_concepts": unique(all_missing),
    "improvement_suggestions": all_suggestions
})

# Export valid JSON
output_path = Path("interview_final_report.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(final_report, f, indent=2, ensure_ascii=False, allow_nan=False)

# Verify exported JSON can be loaded
with output_path.open("r", encoding="utf-8") as f:
    verified_report = json.load(f)

assert verified_report["total_questions"] == len(records)
assert len(verified_report["question_results"]) == len(records)

print("PASS: Enhanced report created and JSON verified.")
print("Questions:", verified_report["total_questions"])
print("Overall score:", verified_report["total_score"], "/", verified_report["total_max_marks"])
print("Overall percentage:", verified_report["percentage"], "%")
print("\nTopic-wise performance:")
for topic, stats in verified_report["topic_summary"].items():
    print(f"- {topic}: {stats['score']}/{stats['max_marks']} ({stats['percentage']}%)")
print("\nJSON file:", output_path.resolve())


PASS: Enhanced report created and JSON verified.
Questions: 6
Overall score: 10.0 / 60.0
Overall percentage: 16.67 %

Topic-wise performance:
- Java: 10.0/30.0 (33.33%)
- OOPs: 0.0/10.0 (0.0%)
- DBMS: 0.0/10.0 (0.0%)
- DSA: 0.0/10.0 (0.0%)

JSON file: /content/interview_final_report.json


In [163]:

!pip -q install reportlab

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, KeepTogether
)
from xml.sax.saxutils import escape
from pathlib import Path

pdf_path = "/content/interview_final_report.pdf"

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    name="ReportTitle", parent=styles["Title"],
    alignment=TA_CENTER, fontSize=20, leading=25,
    textColor=colors.HexColor("#4338CA"), spaceAfter=14
))
styles.add(ParagraphStyle(
    name="Section", parent=styles["Heading2"],
    textColor=colors.HexColor("#4338CA"),
    spaceBefore=12, spaceAfter=7
))
styles.add(ParagraphStyle(
    name="SmallBody", parent=styles["BodyText"],
    fontSize=9, leading=13, spaceAfter=5
))

def safe_text(value):
    if value is None:
        return "Not available"
    if isinstance(value, (list, tuple)):
        return ", ".join(str(x) for x in value) if value else "None recorded"
    text = str(value).strip()
    return escape(text) if text else "Not available"

story = [
    Paragraph("Interview Assessment Report", styles["ReportTitle"]),
    Paragraph("Generated from the current assessment session", styles["SmallBody"]),
    Spacer(1, 0.3 * cm)
]

# Summary
story.append(Paragraph("Overall Performance", styles["Section"]))
summary_data = [
    ["Total Questions", str(final_report.get("total_questions", 0))],
    ["Total Score", f"{final_report.get('total_score', 0)} / {final_report.get('total_max_marks', 0)}"],
    ["Percentage", f"{final_report.get('percentage', 0)}%"]
]
summary_table = Table(summary_data, colWidths=[6*cm, 9*cm])
summary_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#EEF2FF")),
    ("TEXTCOLOR", (0, 0), (-1, -1), colors.HexColor("#1F2937")),
    ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#D1D5DB")),
    ("PADDING", (0, 0), (-1, -1), 8),
    ("VALIGN", (0, 0), (-1, -1), "TOP")
]))
story.append(summary_table)

# Topic-wise performance
story.append(Paragraph("Topic-wise Performance", styles["Section"]))
topic_rows = [["Topic", "Questions", "Score", "Percentage"]]
for topic, stats in final_report.get("topic_summary", {}).items():
    topic_rows.append([
        safe_text(topic),
        str(stats.get("questions", 0)),
        f"{stats.get('score', 0)} / {stats.get('max_marks', 0)}",
        f"{stats.get('percentage', 0)}%"
    ])

topic_table = Table(topic_rows, repeatRows=1, colWidths=[5*cm, 2.5*cm, 4*cm, 3*cm])
topic_table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#4338CA")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#D1D5DB")),
    ("PADDING", (0, 0), (-1, -1), 6),
    ("VALIGN", (0, 0), (-1, -1), "TOP")
]))
story.append(topic_table)

def add_bullet_section(title, items):
    story.append(Paragraph(title, styles["Section"]))
    if not items:
        story.append(Paragraph("No data recorded.", styles["SmallBody"]))
        return
    for item in items:
        if isinstance(item, dict):
            text = " — ".join(
                f"{k.replace('_', ' ').title()}: {v}"
                for k, v in item.items()
            )
        else:
            text = str(item)
        story.append(Paragraph("• " + safe_text(text), styles["SmallBody"]))

add_bullet_section("Overall Strengths", final_report.get("overall_strengths", []))
add_bullet_section("Missing Concepts", final_report.get("overall_missing_concepts", []))
add_bullet_section("Improvement Suggestions", final_report.get("improvement_suggestions", []))

# Question-wise results
story.append(PageBreak())
story.append(Paragraph("Question-wise Assessment", styles["Section"]))

for record in final_report.get("question_results", []):
    evaluation = record.get("evaluation", {})
    block = [
        Paragraph(
            f"Question {record.get('question_number', '?')} — "
            f"{safe_text(record.get('topic'))}",
            styles["Heading3"]
        ),
        Paragraph(safe_text(record.get("question")), styles["SmallBody"]),
        Paragraph(
            f"Difficulty: {safe_text(record.get('difficulty'))} | "
            f"Score: {evaluation.get('score', 0)} / "
            f"{evaluation.get('max_marks', 0)}",
            styles["SmallBody"]
        ),
        Paragraph(
            "<b>Strengths:</b> " +
            safe_text(evaluation.get("strengths", [])),
            styles["SmallBody"]
        ),
        Paragraph(
            "<b>Missing concepts:</b> " +
            safe_text(evaluation.get("missing_concepts", [])),
            styles["SmallBody"]
        ),
        Paragraph(
            "<b>Feedback:</b> " +
            safe_text(evaluation.get("improvement_feedback", "")),
            styles["SmallBody"]
        ),
        Spacer(1, 0.2 * cm)
    ]
    story.append(KeepTogether(block))

def add_page_number(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.setFillColor(colors.grey)
    canvas.drawString(2*cm, 1*cm, "Interview Assessment Report")
    canvas.drawRightString(A4[0] - 2*cm, 1*cm, f"Page {doc.page}")
    canvas.restoreState()

doc = SimpleDocTemplate(
    pdf_path, pagesize=A4,
    rightMargin=2*cm, leftMargin=2*cm,
    topMargin=1.8*cm, bottomMargin=1.8*cm
)
doc.build(story, onFirstPage=add_page_number, onLaterPages=add_page_number)

assert Path(pdf_path).exists() and Path(pdf_path).stat().st_size > 0
print("PASS: PDF generated:", pdf_path)
print("Size:", round(Path(pdf_path).stat().st_size / 1024, 2), "KB")

from google.colab import files
files.download(pdf_path)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 11.3 MB/s eta 0:00:00
PASS: PDF generated: /content/interview_final_report.pdf
Size: 8.03 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [164]:

# Read-only integration-test preflight

required_functions = [
    "evaluate_answer",
    "choose_next_action",
    "generate_next_with_history",
    "run_adaptive_step",
    "append_adaptive_result_to_report",
]

for name in required_functions:
    obj = globals().get(name)
    print(f"{name}: {'AVAILABLE' if callable(obj) else 'MISSING'}")

print("\nCurrent question:", "AVAILABLE" if "current_question" in globals() else "MISSING")
print("Candidate answer:", "AVAILABLE" if "candidate_answer" in globals() else "MISSING")
print("Adaptive result:", "AVAILABLE" if "adaptive_result" in globals() else "MISSING")
print("Assessment report:", len(assessment_report) if "assessment_report" in globals() else "MISSING")
print("Final report:", "AVAILABLE" if isinstance(globals().get("final_report"), dict) else "MISSING")


evaluate_answer: AVAILABLE
choose_next_action: AVAILABLE
generate_next_with_history: AVAILABLE
run_adaptive_step: AVAILABLE
append_adaptive_result_to_report: AVAILABLE

Current question: AVAILABLE
Candidate answer: AVAILABLE
Adaptive result: AVAILABLE
Assessment report: 6
Final report: AVAILABLE


In [165]:

import inspect

for name in [
    "run_adaptive_step",
    "generate_next_with_history",
    "append_adaptive_result_to_report",
]:
    fn = globals()[name]
    print(f"\n{name}{inspect.signature(fn)}")
    print((inspect.getdoc(fn) or "No docstring available").split("\n")[0])



run_adaptive_step(current_question, candidate_answer, question_history, asked_topics=None, resume_text='', job_description='', target_role='Software Developer', experience_level='Beginner')
No docstring available

generate_next_with_history(decision, question_history, resume_text='', job_description='', target_role='Software Developer', experience_level='Beginner')
No docstring available

append_adaptive_result_to_report(assessment_report, current_question, candidate_answer, evaluation)
Append one evaluated adaptive question without overwriting prior records.


In [166]:

import copy

# Snapshot current state
report_count_before = len(assessment_report)
report_snapshot = copy.deepcopy(assessment_report)
final_report_snapshot = copy.deepcopy(final_report)
history_test = copy.deepcopy(question_history)

print("Starting one adaptive integration test...")

try:
    integration_result = run_adaptive_step(
        current_question=current_question,
        candidate_answer=candidate_answer,
        question_history=history_test,
        asked_topics=None,
        resume_text="",
        job_description="",
        target_role="Software Developer",
        experience_level="Beginner"
    )

    print("\nReturned type:", type(integration_result).__name__)

    if isinstance(integration_result, dict):
        print("Returned keys:", list(integration_result.keys()))
        print("Evaluation present:", integration_result.get("evaluation") is not None)
        print("Decision present:", integration_result.get("decision") is not None)
        print("Next question present:", integration_result.get("next_question") is not None)

        next_q = integration_result.get("next_question")
        if next_q is not None:
            print("Next question:", getattr(next_q, "question", str(next_q)))
            print("Next topic:", getattr(next_q, "topic", "Unavailable"))
            print("Next difficulty:", getattr(next_q, "difficulty", "Unavailable"))

    else:
        print("Result:", integration_result)

finally:
    # Confirm existing report was not modified
    assert len(assessment_report) == report_count_before, \
        "Report length changed during integration test!"
    assert assessment_report == report_snapshot, \
        "Existing assessment records changed!"
    assert final_report == final_report_snapshot, \
        "Existing final report changed!"

print("\nPASS: Existing report and final_report remain unchanged.")
print("Original question history length:", len(question_history))
print("Test history length:", len(history_test))


Starting one adaptive integration test...


BadRequestError: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}

In [170]:

import copy

_report_before = copy.deepcopy(assessment_report)
_final_before = copy.deepcopy(final_report)
_history_before = copy.deepcopy(question_history)

try:
    integration_result = run_adaptive_step(
        current_question=current_question,
        candidate_answer=candidate_answer,
        question_history=copy.deepcopy(question_history),
        asked_topics=None,
        resume_text="",
        job_description="",
        target_role="Software Developer",
        experience_level="Beginner"
    )

    print("Returned type:", type(integration_result).__name__)

    assert isinstance(integration_result, dict), \
        "Expected run_adaptive_step to return a dictionary."

    print("Returned keys:", list(integration_result.keys()))

    evaluation = integration_result.get("evaluation")
    decision = integration_result.get("decision")
    next_question = integration_result.get("next_question")

    assert evaluation is not None, "Evaluation missing."
    assert decision is not None, "Adaptive decision missing."
    assert next_question is not None, "Next question missing."

    print("\nEvaluation score:",
          evaluation.get("score") if isinstance(evaluation, dict)
          else getattr(evaluation, "score", "Unavailable"))

    print("Decision:", decision)
    print("Next question:", getattr(next_question, "question", "Unavailable"))
    print("Next topic:", getattr(next_question, "topic", "Unavailable"))
    print("Next difficulty:", getattr(next_question, "difficulty", "Unavailable"))

    print("\nPASS: Evaluation → decision → next question completed.")

finally:
    assert assessment_report == _report_before, \
        "Assessment report changed unexpectedly."
    assert final_report == _final_before, \
        "Final report changed unexpectedly."
    assert question_history == _history_before, \
        "Original question history changed unexpectedly."

print("PASS: Existing report and original history remain unchanged.")


Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [171]:

import inspect

print("=== CURRENT QUESTION ===")
print("Question:", current_question.question)
print("Topic:", current_question.topic)
print("Difficulty:", current_question.difficulty)

print("\n=== ADAPTIVE DECISION FUNCTION ===")
print(inspect.getsource(choose_next_action))

print("\n=== ADAPTIVE WORKFLOW FUNCTION ===")
print(inspect.getsource(run_adaptive_step))


=== CURRENT QUESTION ===
Question: Explain the difference between an ArrayList and a LinkedList in Java.
Topic: Java
Difficulty: easy

=== ADAPTIVE DECISION FUNCTION ===
def choose_next_action(question, evaluation, asked_topics=None):
    asked_topics = asked_topics or []

    # Support both dictionaries and Pydantic objects
    if hasattr(evaluation, "model_dump"):
        evaluation = evaluation.model_dump()
    elif not isinstance(evaluation, dict):
        evaluation = vars(evaluation)

    if hasattr(question, "model_dump"):
        question = question.model_dump()
    elif not isinstance(question, dict):
        question = vars(question)

    score = float(evaluation.get("score", 0))
    max_marks = float(evaluation.get("max_marks", 0))
    percentage = score / max_marks * 100 if max_marks else 0

    difficulty = question.get("difficulty", "easy").lower()

    if percentage >= 80:
        next_difficulty = {
            "easy": "medium",
            "medium": "hard",
           

In [172]:

import inspect

print(inspect.getsource(generate_next_with_history))


def generate_next_with_history(
    decision,
    question_history,
    resume_text="",
    job_description="",
    target_role="Software Developer",
    experience_level="Beginner"
):
    topic = decision["current_topic"]
    difficulty = decision["next_difficulty"]


    history_lines = []

    for item in question_history:
        if isinstance(item, dict):
            history_lines.append(
                f"- Question: {item['question']}\n"
                f"  Concept: {item.get('concept', 'Not recorded')}\n"
                f"  Tested angle: {item.get('angle', 'Not recorded')}"
            )
        else:
            # Supports old history entries that are plain strings
            history_lines.append(f"- Question: {item}")

    history_text = "\n".join(history_lines)


    context = f"""
Adaptive interview.
Topic: {topic}
Target difficulty: {difficulty}
Previous score: {decision['percentage']}%
Missing concepts: {decision['missing_concepts']}

Previously asked questions:
{histor

In [169]:

import copy
import traceback

# Preserve existing records and history
_report_before = copy.deepcopy(assessment_report)
_final_before = copy.deepcopy(final_report)
_history_before = copy.deepcopy(question_history)

try:
    evaluation_test = evaluate_answer(
        current_question,
        candidate_answer
    )

    print("PASS: evaluate_answer succeeded.")
    print("Score:", evaluation_test["score"], "/", evaluation_test["max_marks"])
    print("Criteria returned:", len(evaluation_test["criteria_scores"]))
    print("Strengths:", evaluation_test["strengths"])
    print("Missing concepts:", evaluation_test["missing_concepts"])

except Exception as e:
    print("FAIL:", type(e).__name__)
    print("Error:", str(e))
    traceback.print_exc(limit=2)

finally:
    assert assessment_report == _report_before, "Assessment report changed!"
    assert final_report == _final_before, "Final report changed!"
    assert question_history == _history_before, "Original history changed!"

print("\nPASS: Existing report and question history remain unchanged.")


PASS: evaluate_answer succeeded.
Score: 10.0 / 10.0
Criteria returned: 3
Strengths: ['Accurate performance comparison', 'Clear memory overhead explanation', 'Well-structured explanation']
Missing concepts: []

PASS: Existing report and question history remain unchanged.


In [167]:

import inspect

print("=== evaluate_answer source ===")
print(inspect.getsource(evaluate_answer))

print("\n=== Relevant schema/config variables ===")
for name in [
    "evaluation_schema",
    "evaluation_response_schema",
    "evaluation_json_schema",
    "EvaluationResult",
    "RubricCriterion",
    "MODEL_NAME",
]:
    if name in globals():
        value = globals()[name]
        print(f"\n{name}:")
        print(value if name == "MODEL_NAME" else repr(value)[:5000])


=== evaluate_answer source ===
def evaluate_answer(question, candidate_answer):
    if not isinstance(candidate_answer, str) or not candidate_answer.strip():
        raise ValueError("Candidate answer cannot be empty.")

    rubric = question.rubric
    if not rubric:
        raise ValueError("Question must contain at least one rubric criterion.")

    total_weight = sum(item.weight for item in rubric)
    if total_weight <= 0:
        raise ValueError("Rubric weights must add up to a positive value.")

    max_marks = float(question.max_marks)

    # Allocate question marks proportionally across rubric criteria.
    criterion_limits = [
        max_marks * item.weight / total_weight
        for item in rubric
    ]

    criterion_names = [item.name for item in rubric]

    schema = {
        "type": "object",
        "properties": {
            "criteria_scores": {
                "type": "array",
                "items": {
                    "type": "object",
                    "pr

In [168]:

diagnostic_schema = {
    "type": "object",
    "properties": {
        "ok": {"type": "boolean"},
        "message": {"type": "string"}
    },
    "required": ["ok", "message"],
    "additionalProperties": False
}

try:
    diagnostic_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": 'Return {"ok": true, "message": "schema test passed"}.'
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "schema_diagnostic",
                "strict": True,
                "schema": diagnostic_schema
            }
        },
        max_completion_tokens=256,
        temperature=0
    )

    print("Diagnostic response:")
    print(diagnostic_response.choices[0].message.content)
    print("PASS: Minimal strict JSON schema request succeeded.")

except Exception as e:
    print("Diagnostic failed:", type(e).__name__)
    print(str(e))


Diagnostic response:
{"ok":true,"message":"schema test passed"}
PASS: Minimal strict JSON schema request succeeded.


In [151]:
[name for name in globals()
 if any(word in name.lower()
        for word in ["report", "summary", "final_score"])]

['assessment_report', 'final_report']

In [144]:
print("Tracked questions:", len(question_history))
print("Threshold:", SEMANTIC_DUPLICATE_THRESHOLD)

assert len(question_history) == 8

for i, item in enumerate(question_history, start=1):
    question = (
        item.get("question", "")
        if isinstance(item, dict)
        else item.question if hasattr(item, "question")
        else str(item)
    )
    print(f"{i}. {question[:150]}")

print("\nPASS: Existing question history is readable.")

Tracked questions: 8
Threshold: 0.78
1. What is a wrapper class in Java and provide an example of one?
2. Which wrapper class in Java is used to represent a boolean value, and what is its primary purpose?
3. What is the default value of a Boolean object in Java when it is declared but not initialized?
4. Is the Boolean wrapper class immutable in Java? Explain why or why not.
5. Explain the difference between an ArrayList and a LinkedList in Java.
6. Consider Java's LinkedList implementation. Explain why each node incurs additional memory overhead compared to an array element in an ArrayList, estim
7. Consider Java's java.util.LinkedList implementation. Suppose you need to store 10 million elements of type Integer. Estimate the total memory consumpt
8. Suppose you need to maintain a list of 10 million integers that will be frequently updated (insertions and deletions) and also require occasional rand

PASS: Existing question history is readable.


In [145]:
# Check semantic similarity among the existing questions
for i, item in enumerate(question_history):
    q1 = item.get("question", "") if isinstance(item, dict) else str(item)

    for j in range(i + 1, len(question_history)):
        item2 = question_history[j]
        q2 = item2.get("question", "") if isinstance(item2, dict) else str(item2)

        result = find_semantic_duplicate(
            q1,
            [q2],
            threshold=SEMANTIC_DUPLICATE_THRESHOLD
        )

        if result["is_duplicate"]:
            print(
                f"Potential duplicate: Q{i+1} and Q{j+1} "
                f"(similarity={result['similarity']})"
            )

print("Existing-history similarity audit complete.")

Existing-history similarity audit complete.


In [123]:
import inspect
print(inspect.signature(generate_next_with_history))

(decision, question_history, resume_text='', job_description='', target_role='Software Developer', experience_level='Beginner')


In [124]:
import inspect

print("Decision variable exists:", "decision" in globals())
if "decision" in globals():
    print("Decision type:", type(decision).__name__)
    print("Decision value:", decision)

print("History count:", len(question_history))
print("Function signature:", inspect.signature(generate_next_with_history))

Decision variable exists: True
Decision type: dict
Decision value: {'action': 'INCREASE_DIFFICULTY', 'current_topic': 'Java', 'next_difficulty': 'hard', 'percentage': 100.0, 'strengths': ['Accurate description of node structure and overhead', 'Realistic overhead estimate', 'Clear discussion of memory, GC, and cache effects'], 'missing_concepts': [], 'asked_topics': ['Java'], 'reason': 'Strong answer; test deeper understanding.'}
History count: 7
Function signature: (decision, question_history, resume_text='', job_description='', target_role='Software Developer', experience_level='Beginner')


In [126]:
from unittest.mock import Mock
from types import SimpleNamespace

# Independent copy: original history won't be changed
test_history = [dict(item) if isinstance(item, dict) else item
                for item in question_history]
history_before = [dict(item) if isinstance(item, dict) else item
                  for item in test_history]

duplicate_question = SimpleNamespace(
    question="How does Java ArrayList differ from LinkedList in terms of storage and performance?",
    topic="Java"
)

mock_generate = Mock(
    return_value=SimpleNamespace(questions=[duplicate_question])
)
mock_validate = Mock(
    side_effect=AssertionError("Technical validation should not run!")
)
mock_metadata = Mock(
    side_effect=AssertionError("Metadata extraction should not run!")
)

# Save original functions
original_generate = generate_questions
original_validate = validate_question_technical_accuracy
original_metadata = extract_question_metadata

try:
    generate_questions = mock_generate
    validate_question_technical_accuracy = mock_validate
    extract_question_metadata = mock_metadata

    try:
        generate_next_with_history(decision, test_history)
    except RuntimeError as e:
        print("Expected retry-limit result:", e)

    print("\nGeneration calls:", mock_generate.call_count)
    print("Technical validation calls:", mock_validate.call_count)
    print("Metadata extraction calls:", mock_metadata.call_count)

    assert mock_generate.call_count == 3
    assert mock_validate.call_count == 0
    assert mock_metadata.call_count == 0
    assert test_history == history_before

    print("PASS: 3 duplicate attempts detected.")
    print("PASS: Technical validation skipped.")
    print("PASS: Metadata extraction skipped.")
    print("PASS: Test history unchanged.")
    print("\nAll semantic duplicate mock tests passed!")

finally:
    generate_questions = original_generate
    validate_question_technical_accuracy = original_validate
    extract_question_metadata = original_metadata

Attempt 1: semantic duplicate (similarity=0.8658, matched='Explain the difference between an ArrayList and a LinkedList in Java.'). Retrying...
Attempt 2: semantic duplicate (similarity=0.8658, matched='Explain the difference between an ArrayList and a LinkedList in Java.'). Retrying...
Attempt 3: semantic duplicate (similarity=0.8658, matched='Explain the difference between an ArrayList and a LinkedList in Java.'). Retrying...
Expected retry-limit result: Could not generate a unique, technically accurate question after 3 attempts. Please retry.

Generation calls: 3
Technical validation calls: 0
Metadata extraction calls: 0
PASS: 3 duplicate attempts detected.
PASS: Technical validation skipped.
PASS: Metadata extraction skipped.
PASS: Test history unchanged.

All semantic duplicate mock tests passed!


In [120]:
import inspect

source = inspect.getsource(generate_next_with_history)

semantic_pos = source.find("Semantic duplicate check")
exact_pos = source.find("exact duplicate")
technical_pos = source.find("Technical accuracy check")

print("Semantic check present:", semantic_pos != -1)
print("Technical check present:", technical_pos != -1)
print("Semantic check before technical check:",
      semantic_pos != -1 and technical_pos != -1 and semantic_pos < technical_pos)

Semantic check present: True
Technical check present: True
Semantic check before technical check: True


In [121]:
import inspect

source = inspect.getsource(generate_next_with_history)

exact_pos = source.find("exact duplicate")
semantic_pos = source.find("Semantic duplicate check")
technical_pos = source.find("Technical accuracy check")
metadata_pos = source.find("extract_question_metadata")

print("Exact duplicate check:", exact_pos)
print("Semantic duplicate check:", semantic_pos)
print("Technical validation:", technical_pos)
print("Metadata extraction:", metadata_pos)

assert exact_pos != -1
assert semantic_pos != -1
assert technical_pos != -1
assert metadata_pos != -1

assert exact_pos < semantic_pos < technical_pos < metadata_pos

print("\nExecution order verified!")

Exact duplicate check: 4030
Semantic duplicate check: 4406
Technical validation: 4963
Metadata extraction: 5323

Execution order verified!


In [118]:
test_duplicate = find_semantic_duplicate(
    "How does Java ArrayList differ from LinkedList in terms of storage and performance?",
    question_history,
    threshold=SEMANTIC_DUPLICATE_THRESHOLD
)

test_new = find_semantic_duplicate(
    "What is method overloading in Java?",
    question_history,
    threshold=SEMANTIC_DUPLICATE_THRESHOLD
)

print("Similar question detected:", test_duplicate["is_duplicate"])
print("Similarity:", test_duplicate["similarity"])
print("Matched question:", test_duplicate["matched_question"])

print("\nUnrelated question detected as duplicate:", test_new["is_duplicate"])

assert test_duplicate["is_duplicate"] is True
assert test_new["is_duplicate"] is False

print("\nLocal semantic checks passed.")

Similar question detected: True
Similarity: 0.8658
Matched question: Explain the difference between an ArrayList and a LinkedList in Java.

Unrelated question detected as duplicate: False

Local semantic checks passed.


In [117]:
print("Model loaded:", "similarity_model" in globals())
print("Threshold:", SEMANTIC_DUPLICATE_THRESHOLD)
print("History count:", len(question_history))

assert "similarity_model" in globals()
assert SEMANTIC_DUPLICATE_THRESHOLD == 0.80
assert len(question_history) == 7

print("Pre-check passed.")

Model loaded: True
Threshold: 0.8
History count: 7
Pre-check passed.


In [103]:

decision = choose_next_action(
    current_question,
    evaluation,
    asked_topics=["Java"]
)

print("Adaptive decision:")
print(decision)


Adaptive decision:
{'action': 'INCREASE_DIFFICULTY', 'current_topic': 'Java', 'next_difficulty': 'medium', 'percentage': 85.0, 'strengths': ['Accurate description of internal structures', 'Clear explanation of time complexities for access, insertion, and deletion', 'Concise comparison of use cases'], 'missing_concepts': ['Memory overhead of LinkedList nodes', 'Cache locality advantage of ArrayList', 'Amortized O(1) add at end for ArrayList', 'O(1) add/remove at ends for LinkedList'], 'asked_topics': ['Java'], 'reason': 'Strong answer; test deeper understanding.'}


In [104]:

next_question = generate_next_with_history(
    decision=decision,
    question_history=question_history,
    resume_text=globals().get("resume_text", ""),
    job_description=globals().get("job_description", ""),
    target_role="Software Developer",
    experience_level="Beginner"
)

print("NEXT QUESTION:")
print(next_question.question)
print("TOPIC:", next_question.topic)
print("DIFFICULTY:", next_question.difficulty)
print("HISTORY COUNT:", len(question_history))


Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [85]:
print("Total questions:", len(question_history))
print("Latest history entry:")
print(question_history[-1])

Total questions: 4
Latest history entry:
{'question': 'Is the Boolean wrapper class immutable in Java? Explain why or why not.', 'concept': 'Boolean wrapper class', 'angle': 'Immutability and design rationale'}


In [63]:

def validate_question_technical_accuracy(question):
    """
    Use the same Groq model to review a generated question for
    factual errors, misleading premises, and incorrect API behavior.
    """
    if hasattr(question, "model_dump"):
        question = question.model_dump()

    question_text = question.get("question", "")

    prompt = f"""
    You are a strict Java technical reviewer.

    Review this interview question for technical correctness:

    {question_text}

    Check whether it:
    1. Contains any false or misleading technical premise.
    2. Describes Java exception handling and APIs correctly.
    3. Requests something impossible or incorrectly described.
    4. Is clear and answerable.

    Important Java rule:
    In ordinary try-finally, an exception thrown by finally can mask
    the exception thrown in try. It is NOT automatically added to
    getSuppressed(). Automatic suppression is a try-with-resources behavior.
    Manual suppression can be implemented using addSuppressed().

    Return JSON with exactly these fields:
    {{
      "is_accurate": true,
      "issues": [],
      "explanation": "Short explanation"
    }}

    Set is_accurate to false if the question contains a factual error
    or a misleading premise. Do not reject a question merely because
    it discusses a difficult concept.
    """

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You review Java interview questions for technical accuracy."
            },
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    import json
    return json.loads(response.choices[0].message.content)


In [127]:
from unittest.mock import Mock
from types import SimpleNamespace

# Snapshot history so we can confirm only one record is appended
test_history = [
    dict(item) if isinstance(item, dict) else item
    for item in question_history
]
history_before = [
    dict(item) if isinstance(item, dict) else item
    for item in test_history
]

# A new question that is not in the current history
new_question = SimpleNamespace(
    question="Explain the difference between method overloading and method overriding in Java.",
    topic="Java"
)

# Mock all external/API-dependent functions
mock_generate = Mock(
    return_value=SimpleNamespace(questions=[new_question])
)
mock_validate = Mock(
    return_value={
        "is_accurate": True,
        "issues": [],
        "explanation": "The question is technically accurate and answerable."
    }
)
mock_metadata = Mock(
    return_value={
        "concept": "Overloading vs Overriding",
        "angle": "Compare compile-time and runtime polymorphism"
    }
)

original_generate = generate_questions
original_validate = validate_question_technical_accuracy
original_metadata = extract_question_metadata

try:
    generate_questions = mock_generate
    validate_question_technical_accuracy = mock_validate
    extract_question_metadata = mock_metadata

    result = generate_next_with_history(decision, test_history)

    print("Generated question:", result.question)
    print("Generation calls:", mock_generate.call_count)
    print("Technical validation calls:", mock_validate.call_count)
    print("Metadata extraction calls:", mock_metadata.call_count)
    print("History entries before:", len(history_before))
    print("History entries after:", len(test_history))
    print("Last history record:", test_history[-1])

    assert mock_generate.call_count == 1
    assert mock_validate.call_count == 1
    assert mock_metadata.call_count == 1
    assert len(test_history) == len(history_before) + 1
    assert test_history[-1]["concept"] == "Overloading vs Overriding"

    print("\nPASS: Successful non-duplicate path verified.")

finally:
    generate_questions = original_generate
    validate_question_technical_accuracy = original_validate
    extract_question_metadata = original_metadata

Question passed validation on attempt 1.
Generated question: Explain the difference between method overloading and method overriding in Java.
Generation calls: 1
Technical validation calls: 1
Metadata extraction calls: 1
History entries before: 7
History entries after: 8
Last history record: {'question': 'Explain the difference between method overloading and method overriding in Java.', 'concept': 'Overloading vs Overriding', 'angle': 'Compare compile-time and runtime polymorphism'}

PASS: Successful non-duplicate path verified.


In [128]:
print("generate_questions restored:", generate_questions is original_generate)
print("validation restored:", validate_question_technical_accuracy is original_validate)
print("metadata restored:", extract_question_metadata is original_metadata)

print("Original history count:", len(question_history))
print("Mock test history count:", len(test_history))

assert generate_questions is original_generate
assert validate_question_technical_accuracy is original_validate
assert extract_question_metadata is original_metadata
assert len(question_history) == 7

print("\nAll restoration checks passed.")

generate_questions restored: True
validation restored: True
metadata restored: True
Original history count: 7
Mock test history count: 8

All restoration checks passed.


In [129]:
result = generate_next_with_history(decision, question_history)

print("\nGenerated question:")
print(result.question)

print("\nTopic:", result.topic)
print("History count:", len(question_history))
print("Last history record:", question_history[-1])

Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [130]:
last_question = question_history[-1]["question"]

print("Last saved question:")
print(last_question)

print("\nSemantic duplicate recheck:")
print(
    find_semantic_duplicate(
        last_question,
        question_history[:-1],
        threshold=SEMANTIC_DUPLICATE_THRESHOLD
    )
)

Last saved question:
Suppose you need to maintain a list of 10 million integers that will be frequently updated (insertions and deletions) and also require occasional random access. Discuss the trade‑offs between using a java.util.LinkedList<Integer> and a java.util.ArrayList<Integer> in terms of memory consumption, insertion/deletion performance, and random access performance. Then design a custom data structure that combines the advantages of both, describing its internal representation and how it addresses the trade‑offs. Provide pseudocode for insertion, deletion, and random access operations, and analyze the time and space complexity.

Semantic duplicate recheck:
{'is_duplicate': False, 'similarity': 0.7872, 'matched_question': "Consider Java's java.util.LinkedList implementation. Suppose you need to store 10 million elements of type Integer. Estimate the total memory consumption of the LinkedList, including node overhead, and compare it to an ArrayList storing the same elements. 

In [131]:
SEMANTIC_DUPLICATE_THRESHOLD = 0.78

last_question = question_history[-1]["question"]

check = find_semantic_duplicate(
    last_question,
    question_history[:-1],
    threshold=SEMANTIC_DUPLICATE_THRESHOLD
)

print("Threshold:", SEMANTIC_DUPLICATE_THRESHOLD)
print("Duplicate detected:", check["is_duplicate"])
print("Similarity:", check["similarity"])
print("Matched question:", check["matched_question"])

assert check["is_duplicate"] is True
print("\nPASS: Similar question now gets rejected.")

Threshold: 0.78
Duplicate detected: True
Similarity: 0.7872
Matched question: Consider Java's java.util.LinkedList implementation. Suppose you need to store 10 million elements of type Integer. Estimate the total memory consumption of the LinkedList, including node overhead, and compare it to an ArrayList storing the same elements. Discuss the performance implications for insertion, deletion, and traversal in both structures. Finally, propose a strategy to reduce memory overhead while preserving the ability to perform frequent insertions and deletions, and explain how your solution mitigates the identified issues.

PASS: Similar question now gets rejected.


In [132]:
duplicate_test = find_semantic_duplicate(
    "Explain how LinkedList and ArrayList differ in memory usage and performance.",
    question_history,
    threshold=SEMANTIC_DUPLICATE_THRESHOLD
)

new_topic_test = find_semantic_duplicate(
    "Explain method overloading and method overriding in Java.",
    question_history,
    threshold=SEMANTIC_DUPLICATE_THRESHOLD
)

print("Related question rejected:", duplicate_test["is_duplicate"])
print("Related question similarity:", duplicate_test["similarity"])

print("\nUnrelated question rejected:", new_topic_test["is_duplicate"])
print("Unrelated question similarity:", new_topic_test["similarity"])

assert duplicate_test["is_duplicate"] is True
assert new_topic_test["is_duplicate"] is False

print("\nPASS: Both regression tests passed.")

Related question rejected: True
Related question similarity: 0.8385

Unrelated question rejected: False
Unrelated question similarity: 0.4911

PASS: Both regression tests passed.


In [134]:

from unittest.mock import Mock

# Save original functions
_original_generate = generate_questions
_original_validate = validate_question_technical_accuracy
_original_metadata = extract_question_metadata

test_history = [
    {
        "question": "Explain the difference between ArrayList and LinkedList in Java.",
        "concept": "Collections",
        "angle": "Memory and performance"
    }
]

duplicate_question = Mock()
duplicate_question.question = (
    "Compare Java ArrayList and LinkedList for memory usage and performance."
)
duplicate_question.topic = "Java"

fresh_question = Mock()
fresh_question.question = (
    "Explain method overloading and method overriding in Java."
)
fresh_question.topic = "Java"

# First generation is a duplicate; second is new
generate_questions = Mock(
    side_effect=[
        Mock(questions=[duplicate_question]),
        Mock(questions=[fresh_question])
    ]
)
validate_question_technical_accuracy = Mock(
    return_value={"is_accurate": True, "issues": []}
)
extract_question_metadata = Mock(
    return_value={
        "concept": "Overloading vs Overriding",
        "angle": "Compile-time vs runtime polymorphism"
    }
)

try:
    result = generate_next_with_history(
        decision,
        test_history.copy()
    )

    assert generate_questions.call_count == 2
    assert validate_question_technical_accuracy.call_count == 1
    assert extract_question_metadata.call_count == 1

    print("PASS: Duplicate was retried.")
    print("PASS: Fresh question was accepted.")
    print("PASS: Validation ran only for the accepted question.")
    print("Generation attempts:", generate_questions.call_count)
    print("Accepted question:", result["question"])

finally:
    # Restore original functions even if the test fails
    generate_questions = _original_generate
    validate_question_technical_accuracy = _original_validate
    extract_question_metadata = _original_metadata


Question passed validation on attempt 1.


AssertionError: 

In [135]:
print("Generation calls:", generate_questions.call_count)
print("Validation calls:", validate_question_technical_accuracy.call_count)
print("Metadata calls:", extract_question_metadata.call_count)
print("Threshold:", SEMANTIC_DUPLICATE_THRESHOLD)

print("\nDuplicate test question:")
print(duplicate_question.question)

print("\nSemantic score against test history:")
print(
    find_semantic_duplicate(
        duplicate_question.question,
        test_history,
        threshold=SEMANTIC_DUPLICATE_THRESHOLD
    )
)

AttributeError: 'function' object has no attribute 'call_count'

In [138]:

from unittest.mock import Mock

test_history = [{
    "question": "Old question",
    "concept": "Test concept",
    "angle": "Test angle"
}]

duplicate_q = Mock()
duplicate_q.question = "First generated question"
duplicate_q.topic = "Java"

fresh_q = Mock()
fresh_q.question = "Second generated question"
fresh_q.topic = "Java"

gen_mock = Mock(side_effect=[
    Mock(questions=[duplicate_q]),
    Mock(questions=[fresh_q])
])
semantic_mock = Mock(side_effect=[
    {
        "is_duplicate": True,
        "similarity": 0.91,
        "matched_question": "Old question"
    },
    {
        "is_duplicate": False,
        "similarity": 0.20,
        "matched_question": None
    }
])
validate_mock = Mock(
    return_value={"is_accurate": True, "issues": []}
)
metadata_mock = Mock(return_value={
    "concept": "Test concept",
    "angle": "New angle"
})

_originals = (
    generate_questions,
    find_semantic_duplicate,
    validate_question_technical_accuracy,
    extract_question_metadata
)

try:
    generate_questions = gen_mock
    find_semantic_duplicate = semantic_mock
    validate_question_technical_accuracy = validate_mock
    extract_question_metadata = metadata_mock

    result = generate_next_with_history(decision, test_history)

    print("Generation calls:", gen_mock.call_count)
    print("Semantic checks:", semantic_mock.call_count)
    print("Validation calls:", validate_mock.call_count)
    print("Metadata calls:", metadata_mock.call_count)
    print("Accepted question:", result.question)

    assert gen_mock.call_count == 2
    assert semantic_mock.call_count == 2
    assert validate_mock.call_count == 1
    assert metadata_mock.call_count == 1
    assert result is fresh_q

    print("\nPASS: Duplicate retry flow verified.")

finally:
    (
        generate_questions,
        find_semantic_duplicate,
        validate_question_technical_accuracy,
        extract_question_metadata
    ) = _originals


Attempt 1: semantic duplicate (similarity=0.91, matched='Old question'). Retrying...
Question passed validation on attempt 2.
Generation calls: 2
Semantic checks: 2
Validation calls: 1
Metadata calls: 1
Accepted question: Second generated question

PASS: Duplicate retry flow verified.


In [139]:

# Final integration test — real generation
import copy

history_backup = copy.deepcopy(question_history)
history_before = len(question_history)

try:
    result = generate_next_with_history(
        decision,
        question_history,
        resume_text="",
        job_description="",
        target_role="Software Developer",
        experience_level="Beginner"
    )

    assert result is not None
    assert hasattr(result, "question")
    assert result.question.strip()

    # Verify the accepted question was added to history
    assert len(question_history) == history_before + 1
    assert question_history[-1]["question"] == result.question

    # Check accepted question against earlier history
    final_check = find_semantic_duplicate(
        result.question,
        question_history[:-1],
        threshold=SEMANTIC_DUPLICATE_THRESHOLD
    )

    print("\nPASS: Real generation returned a question.")
    print("PASS: Accepted question was added to history.")
    print("Question:", result.question)
    print("Concept:", question_history[-1]["concept"])
    print("Angle:", question_history[-1]["angle"])
    print("Closest prior similarity:", final_check["similarity"])
    print("Duplicate of earlier question:", final_check["is_duplicate"])

    assert not final_check["is_duplicate"], (
        "Accepted question matches earlier history; review threshold or logic."
    )
    print("\nALL FINAL INTEGRATION CHECKS PASSED.")

finally:
    question_history[:] = history_backup


Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

BadRequestError: Error code: 400 - {'error': {'message': "Generated JSON does not match the expected schema. Please adjust your prompt. See 'failed_generation' for more details. Error: jsonschema: '/questions/0' does not validate with /properties/questions/items/$ref/required: missing properties: 'rubric', 'max_marks'", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': '{"questions":[{"question":"You need to store 10\u202fmillion integers in a list that supports frequent insertions and deletions. You consider using a LinkedList<Integer> but are concerned about memory overhead. Suppose you design a custom singly linked list where each node stores a primitive int value and a reference to the next node. Estimate the memory consumption of this structure for 10\u202fmillion elements, assuming 8\u202fbytes per reference and 4\u202fbytes per int, plus an object header overhead of 12\u202fbytes per node. Compare this to the memory consumption of an ArrayList

In [137]:
import inspect

print(inspect.getsource(generate_next_with_history))

def generate_next_with_history(
    decision,
    question_history,
    resume_text="",
    job_description="",
    target_role="Software Developer",
    experience_level="Beginner"
):
    topic = decision["current_topic"]
    difficulty = decision["next_difficulty"]


    history_lines = []

    for item in question_history:
        if isinstance(item, dict):
            history_lines.append(
                f"- Question: {item['question']}\n"
                f"  Concept: {item.get('concept', 'Not recorded')}\n"
                f"  Tested angle: {item.get('angle', 'Not recorded')}"
            )
        else:
            # Supports old history entries that are plain strings
            history_lines.append(f"- Question: {item}")

    history_text = "\n".join(history_lines)


    context = f"""
Adaptive interview.
Topic: {topic}
Target difficulty: {difficulty}
Previous score: {decision['percentage']}%
Missing concepts: {decision['missing_concepts']}

Previously asked questions:
{histor

In [136]:

from unittest.mock import Mock

test_history = [
    {
        "question": "Explain the difference between ArrayList and LinkedList in Java.",
        "concept": "Collections",
        "angle": "Memory and performance"
    }
]

duplicate_q = Mock()
duplicate_q.question = (
    "Compare Java ArrayList and LinkedList for memory usage and performance."
)
duplicate_q.topic = "Java"

fresh_q = Mock()
fresh_q.question = (
    "Explain method overloading and method overriding in Java."
)
fresh_q.topic = "Java"

gen_mock = Mock(side_effect=[
    Mock(questions=[duplicate_q]),
    Mock(questions=[fresh_q])
])
validate_mock = Mock(
    return_value={"is_accurate": True, "issues": []}
)
metadata_mock = Mock(return_value={
    "concept": "Overloading vs Overriding",
    "angle": "Compile-time vs runtime polymorphism"
})

_original_generate = generate_questions
_original_validate = validate_question_technical_accuracy
_original_metadata = extract_question_metadata

try:
    generate_questions = gen_mock
    validate_question_technical_accuracy = validate_mock
    extract_question_metadata = metadata_mock

    result = generate_next_with_history(decision, test_history)

    print("Generation calls:", gen_mock.call_count)
    print("Validation calls:", validate_mock.call_count)
    print("Metadata calls:", metadata_mock.call_count)
    print("Accepted question:", result["question"])

    assert gen_mock.call_count == 2, "Expected one duplicate retry"
    assert validate_mock.call_count == 1
    assert metadata_mock.call_count == 1
    print("PASS: Retry flow verified.")

finally:
    generate_questions = _original_generate
    validate_question_technical_accuracy = _original_validate
    extract_question_metadata = _original_metadata


Question passed validation on attempt 1.
Generation calls: 1
Validation calls: 1
Metadata calls: 1


TypeError: 'Mock' object is not subscriptable

In [64]:

accuracy_result = validate_question_technical_accuracy(
    adaptive_output["next_question"]
)

print("Technically accurate:", accuracy_result["is_accurate"])
print("Issues:", accuracy_result["issues"])
print("Explanation:", accuracy_result["explanation"])


NameError: name 'adaptive_output' is not defined

In [71]:

# Generate using the strong-answer decision from Step 6
next_q3 = generate_next_with_history(
    decision=decision_2,
    question_history=question_history,
    target_role="Software Developer",
    experience_level="Beginner"
)

print("GENERATED QUESTION:")
print(next_q3.question)

print("\nTOPIC:", next_q3.topic)
print("DIFFICULTY:", next_q3.difficulty)

print("\nUPDATED HISTORY:")
print("Questions tracked:", len(question_history))


Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [72]:
candidate_answer_3 = """
Try-with-resources automatically closes resources after the try block,
even if an exception occurs. The resource must implement AutoCloseable
or Closeable. It avoids manually closing streams in a finally block.

Example:
try (BufferedReader br = new BufferedReader(new FileReader("file.txt"))) {
    System.out.println(br.readLine());
} catch (IOException e) {
    e.printStackTrace();
}
"""

evaluation_3 = evaluate_answer(next_q3, candidate_answer_3)

if hasattr(evaluation_3, "model_dump"):
    evaluation_3 = evaluation_3.model_dump()

print("EVALUATION:", evaluation_3)

decision_3 = choose_next_action(
    next_q3,
    evaluation_3,
    asked_topics=["Java, OOPS", "Java"]
)

print("\nNEXT ADAPTIVE DECISION:", decision_3)

EVALUATION: {'criteria_scores': [{'criterion': 'Definition Accuracy', 'marks_awarded': 0, 'max_marks': 5.0, 'feedback': 'Answer does not define a wrapper class; no correct definition provided.'}, {'criterion': 'Example Correctness', 'marks_awarded': 0, 'max_marks': 3.0, 'feedback': 'Example given is unrelated to wrapper classes; it demonstrates try-with-resources instead.'}, {'criterion': 'Clarity', 'marks_awarded': 2, 'max_marks': 2.0, 'feedback': 'Answer is clear, concise, and well-structured, though it does not address the question.'}], 'improvement_feedback': 'To improve, provide a correct definition of a wrapper class in Java and give a valid example such as Integer or Double. Ensure the answer directly addresses the question.', 'missing_concepts': ['Wrapper class definition', 'Wrapper class example'], 'strengths': ['Clear explanation of try-with-resources'], 'score': 2.0, 'max_marks': 10.0}

NEXT ADAPTIVE DECISION: {'action': 'REVISIT_FUNDAMENTALS', 'current_topic': 'Java', 'next

In [73]:
next_q4 = generate_next_with_history(
    decision=decision_3,
    question_history=question_history,
    target_role="Software Developer",
    experience_level="Beginner"
)

print("NEXT QUESTION:", next_q4.question)
print("TOPIC:", next_q4.topic)
print("DIFFICULTY:", next_q4.difficulty)
print("QUESTIONS TRACKED:", len(question_history))

Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [74]:
print("Total questions:", len(question_history))
print("\nQuestion history:")
for i, q in enumerate(question_history, 1):
    print(f"{i}. {q}")

print("\nLatest decision:")
print(decision_3)

Total questions: 2

Question history:
1. {'question': 'What is a wrapper class in Java and provide an example of one?', 'concept': 'Wrapper class', 'angle': 'Definition and example'}
2. {'question': 'Which wrapper class in Java is used to represent a boolean value, and what is its primary purpose?', 'concept': 'Boolean wrapper class', 'angle': 'Primary purpose of Boolean wrapper'}

Latest decision:
{'action': 'REVISIT_FUNDAMENTALS', 'current_topic': 'Java', 'next_difficulty': 'easy', 'percentage': 20.0, 'strengths': ['Clear explanation of try-with-resources'], 'missing_concepts': ['Wrapper class definition', 'Wrapper class example'], 'asked_topics': ['Java, OOPS', 'Java'], 'reason': 'Low score; check foundational understanding.'}


In [75]:
candidate_answer_4 = """
Resources in try-with-resources are closed in reverse order.
If the try block throws an exception and closing resources also
throw exceptions, the try-block exception remains primary.
Closing exceptions are stored as suppressed exceptions and can
be accessed using getSuppressed(). In a traditional try-finally
block, a finally exception can mask the original exception.
Multiple resources can be declared in one try-with-resources statement.
"""

evaluation_4 = evaluate_answer(next_q4, candidate_answer_4)

if hasattr(evaluation_4, "model_dump"):
    evaluation_4 = evaluation_4.model_dump()

print("SCORE:", evaluation_4["score"], "/", evaluation_4["max_marks"])

decision_4 = choose_next_action(
    next_q4,
    evaluation_4,
    asked_topics=["Java, OOPS", "Java"]
)

print("ADAPTIVE DECISION:", decision_4)

SCORE: 0.0 / 10.0
ADAPTIVE DECISION: {'action': 'REVISIT_FUNDAMENTALS', 'current_topic': 'Java', 'next_difficulty': 'easy', 'percentage': 0.0, 'strengths': [], 'missing_concepts': ['Boolean wrapper class'], 'asked_topics': ['Java, OOPS', 'Java'], 'reason': 'Low score; check foundational understanding.'}


In [76]:
next_q5 = generate_next_with_history(
    decision=decision_4,
    question_history=question_history,
    target_role="Software Developer",
    experience_level="Beginner"
)

print("NEXT QUESTION:", next_q5.question)
print("DIFFICULTY:", next_q5.difficulty)
print("TOTAL QUESTIONS:", len(question_history))

Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 'max_marks'], 'title': 

In [77]:

def run_adaptive_step(
    current_question,
    candidate_answer,
    question_history,
    asked_topics=None,
    resume_text="",
    job_description="",
    target_role="Software Developer",
    experience_level="Beginner"
):
    # 1. Evaluate the candidate's answer
    evaluation = evaluate_answer(current_question, candidate_answer)

    if hasattr(evaluation, "model_dump"):
        evaluation = evaluation.model_dump()

    # 2. Decide the next action and difficulty
    decision = choose_next_action(
        current_question,
        evaluation,
        asked_topics=asked_topics or []
    )

    # 3. Generate the next question using history
    next_question = generate_next_with_history(
        decision=decision,
        question_history=question_history,
        resume_text=resume_text,
        job_description=job_description,
        target_role=target_role,
        experience_level=experience_level
    )

    return {
        "evaluation": evaluation,
        "decision": decision,
        "next_question": next_question,
        "questions_tracked": len(question_history)
    }


In [35]:
schema = QuestionSet.model_json_schema()

print(json.dumps(schema, indent=2))

{
  "$defs": {
    "AssessmentQuestion": {
      "properties": {
        "question": {
          "title": "Question",
          "type": "string"
        },
        "topic": {
          "title": "Topic",
          "type": "string"
        },
        "question_type": {
          "enum": [
            "technical",
            "scenario",
            "problem_solving",
            "behavioral"
          ],
          "title": "Question Type",
          "type": "string"
        },
        "difficulty": {
          "enum": [
            "easy",
            "medium",
            "hard"
          ],
          "title": "Difficulty",
          "type": "string"
        },
        "source_relevance": {
          "items": {
            "type": "string"
          },
          "title": "Source Relevance",
          "type": "array"
        },
        "expected_answer_points": {
          "items": {
            "type": "string"
          },
          "title": "Expected Answer Points",
          "type": 

In [78]:

test_answer = """
A try-finally block can be difficult to manage when multiple
resources must be closed. If closing one resource throws an
exception, the next close statement may never execute, causing
a resource leak. Also, an exception from finally can mask the
original exception from the try block.

Example:

try {
    resourceA.use();
    resourceB.use();
} finally {
    resourceB.close();
    resourceA.close();
}

If resourceB.close() throws an exception, resourceA.close()
may never execute. If the try block also throws an exception,
the finally exception can hide the original error.

Try-with-resources handles this more safely:

try (Resource resourceA = new Resource("A");
     Resource resourceB = new Resource("B")) {
    resourceA.use();
    resourceB.use();
}

Resources close automatically in reverse initialization order:
resourceB first, then resourceA. If the main operation throws
an exception, closing exceptions are preserved as suppressed
exceptions instead of replacing the original exception.
"""
print("Question max marks:", next_q5.max_marks)

if not test_answer.strip():
    print("Answer empty hai — evaluation nahi hogi.")
else:
    adaptive_output = run_adaptive_step(
        current_question=next_q5,
        candidate_answer=test_answer,
        question_history=question_history,
        asked_topics=["Java, OOPS", "Java"],
        target_role="Software Developer",
        experience_level="Beginner"
    )

    print("SCORE:",
          adaptive_output["evaluation"]["score"],
          "/",
          adaptive_output["evaluation"]["max_marks"])

    print("ACTION:", adaptive_output["decision"]["action"])
    print("NEXT DIFFICULTY:",
          adaptive_output["decision"]["next_difficulty"])
    print("NEXT QUESTION:",
          adaptive_output["next_question"].question)
    print("QUESTIONS TRACKED:",
          adaptive_output["questions_tracked"])


Question max marks: 10
Strict mode: {'type': 'json_schema', 'json_schema': {'name': 'question_set', 'strict': True, 'schema': {'$defs': {'AssessmentQuestion': {'properties': {'question': {'title': 'Question', 'type': 'string'}, 'topic': {'title': 'Topic', 'type': 'string'}, 'question_type': {'enum': ['technical', 'scenario', 'problem_solving', 'behavioral'], 'title': 'Question Type', 'type': 'string'}, 'difficulty': {'enum': ['easy', 'medium', 'hard'], 'title': 'Difficulty', 'type': 'string'}, 'source_relevance': {'items': {'type': 'string'}, 'title': 'Source Relevance', 'type': 'array'}, 'expected_answer_points': {'items': {'type': 'string'}, 'title': 'Expected Answer Points', 'type': 'array'}, 'rubric': {'items': {'$ref': '#/$defs/RubricCriterion'}, 'title': 'Rubric', 'type': 'array'}, 'max_marks': {'exclusiveMinimum': 0, 'title': 'Max Marks', 'type': 'integer'}}, 'required': ['question', 'topic', 'question_type', 'difficulty', 'source_relevance', 'expected_answer_points', 'rubric', 

In [79]:
print("QUESTION:")
print(next_q5.question)

print("\nEXPECTED ANSWER POINTS:")
print(next_q5.expected_answer_points)

print("\nRUBRIC:")
for criterion in next_q5.rubric:
    print(criterion.name, "-", criterion.description, "| Weight:", criterion.weight)

QUESTION:
What is the default value of a Boolean object in Java when it is declared but not initialized?

EXPECTED ANSWER POINTS:
['The Boolean wrapper class is a final class that encapsulates a boolean primitive. It is immutable. When declared but not initialized, its default value is null.']

RUBRIC:
Correctness of Information - Accurately states that Boolean is a final immutable wrapper for boolean and that its default value is null when declared but not initialized. | Weight: 70
Completeness - Includes all required elements: class type, immutability, and default value. | Weight: 30


In [80]:
print("Decision topic:", adaptive_output["decision"]["current_topic"])
print("Generated topic:", adaptive_output["next_question"].topic)
print("Question:", adaptive_output["next_question"].question)

Decision topic: Java
Generated topic: Java
Question: Is the Boolean wrapper class immutable in Java? Explain why or why not.


In [81]:
print("CURRENT TOPIC:", adaptive_output["decision"]["current_topic"])
print("NEXT QUESTION TOPIC:", adaptive_output["next_question"].topic)
print("NEXT QUESTION DIFFICULTY:", adaptive_output["next_question"].difficulty)

CURRENT TOPIC: Java
NEXT QUESTION TOPIC: Java
NEXT QUESTION DIFFICULTY: easy


In [82]:

print("QUESTION BEING EVALUATED:")
print(next_q5.question)

print("\nEXPECTED ANSWER POINTS:")
print(next_q5.expected_answer_points)

print("\nRUBRIC:")
for criterion in next_q5.rubric:
    if hasattr(criterion, "model_dump"):
        criterion = criterion.model_dump()
    print(criterion)


QUESTION BEING EVALUATED:
What is the default value of a Boolean object in Java when it is declared but not initialized?

EXPECTED ANSWER POINTS:
['The Boolean wrapper class is a final class that encapsulates a boolean primitive. It is immutable. When declared but not initialized, its default value is null.']

RUBRIC:
{'name': 'Correctness of Information', 'description': 'Accurately states that Boolean is a final immutable wrapper for boolean and that its default value is null when declared but not initialized.', 'weight': 70}
{'name': 'Completeness', 'description': 'Includes all required elements: class type, immutability, and default value.', 'weight': 30}


In [83]:
print("History length:", len(question_history))
for i, q in enumerate(question_history, start=1):
    print(f"{i}. {q}")

History length: 4
1. {'question': 'What is a wrapper class in Java and provide an example of one?', 'concept': 'Wrapper class', 'angle': 'Definition and example'}
2. {'question': 'Which wrapper class in Java is used to represent a boolean value, and what is its primary purpose?', 'concept': 'Boolean wrapper class', 'angle': 'Primary purpose of Boolean wrapper'}
3. {'question': 'What is the default value of a Boolean object in Java when it is declared but not initialized?', 'concept': 'Boolean default value', 'angle': 'Default initialization of Boolean objects'}
4. {'question': 'Is the Boolean wrapper class immutable in Java? Explain why or why not.', 'concept': 'Boolean wrapper class', 'angle': 'Immutability and design rationale'}


In [84]:
print("History length:", len(question_history))
print("Latest question:", question_history[-1])
print("Decision topic:", adaptive_output["decision"]["current_topic"])
print("Generated topic:", adaptive_output["next_question"].topic)

History length: 4
Latest question: {'question': 'Is the Boolean wrapper class immutable in Java? Explain why or why not.', 'concept': 'Boolean wrapper class', 'angle': 'Immutability and design rationale'}
Decision topic: Java
Generated topic: Java
